# BananaLSD — MobileNetV3 dari data mentah sampai evaluasi terkunci

Notebook mandiri untuk **PyTorch FP32**, mengikuti tiga README proyek: preprocessing,
training, dan evaluasi. Bahasa penjelasan: Indonesia. Jalankan sel berurutan.
Output notebook ini adalah implementasi; **belum ada hasil training atau skor test yang diklaim**.

**Alur:** audit/EDA → split 70/15/15 → sanity/smoke → E00–E03 → konfirmasi seed
17/2026 → handoff seed 42 → kalibrasi/policy validation → protocol lock → test →
CI/stress/error analysis/benchmark → laporan. Mode `full` menjalankan delapan run;
mode `smoke` hanya diagnostik dan tidak membuka test.

## Penyesuaian dan keputusan implementasi

| Hal | Implementasi dan alasan |
|---|---|
| `dataset.md` | Komentar RAW/AUGMENTED tertukar. Sumber utama **`raw/train-00000-of-00001.parquet`**, bukan `augmented/`. File sumber tidak diubah. |
| Versi sumber | Revision awal dipin ke `56ad86448f39a228cc2ca72ca6bcf66634a24d7d`, diperiksa melalui API HF pada 7 November 2026. Metadata label tetap dibaca saat eksekusi. |
| Bentuk penyerahan | Package/CLI yang diusulkan README digantikan fungsi dan sel di satu `.ipynb`, sesuai permintaan. Manifest/handoff tetap memakai kontrak README. |
| Label | ID proyek mengikuti urutan metadata `ClassLabel`; tidak mengasumsikan ID alfabet. Local directory memakai nama folder dan mencatat mapping eksplisit. |
| Duplikat | Hash piksel RGB sesudah EXIF; duplikat identik satu label dikonsolidasi. **Penyesuaian data nyata:** seluruh anggota konflik label dikarantina dan perlu review; tidak memilih label pengganti. Pilih `conflict_action='error'` untuk perilaku fail-fast rencana awal. Kandidat pHash tidak dihapus. |
| Grup | Jika grup terverifikasi tersedia, split aproksimasi berstrata dengan grup utuh; integritas grup didahulukan dari rasio persis. Tanpa metadata, independensi foto tidak dapat dipastikan. |
| Raw 224×224 | Resize langsung dipertahankan, normalisasi ImageNet sekali. Tidak menambahkan center crop dari `weights.transforms()`. Padding tidak memulihkan rasio foto asli. |
| Eksperimen | Small V1 utama, Large V2 pembanding; 5 epoch head + maks. 25 fine-tune, semua BN backbone dibekukan. Default online augmentation; tidak menggabungkan published augmented. |
| Evaluasi | Validation digunakan ulang untuk kalibrasi/policy prototipe, sehingga hasil development optimistis. Test dibuka sesudah checkpoint/policy/protokol dibekukan. |
| Support kalibrasi | Gate eksplisit minimal 40 validation total dan 5/kelas; tambahan konservatif proyek agar fitting tidak berjalan pada fixture sangat kecil. |
| Data eksternal | Field/OOD nyata dan quality rule berlabel manusia berstatus `pending_external_data`; tidak dibuat data/label pengganti. Ekspor mobile/quantization tetap ditunda. |

**Cara pakai:** pilih kernel Python ≥3.10 dengan pasangan torch/torchvision kompatibel;
isi `PROJECT_ROOT` bila auto-discovery tidak cocok. Ubah `RUN_MODE='smoke'` untuk uji cepat,
kemudian gunakan direktori output baru saat menjalankan `full`. GPU dianjurkan untuk delapan run.
Notebook tidak otomatis meng-upgrade paket. Semua artefak ditulis di `outputs/bananalsd_notebook_device_auto_v2/`.


### Temuan audit data saat notebook disusun

Audit langsung pada Parquet revision yang dipin (SHA-256 file
`e9abdeb7f9b08cc859ffbed2f7e90f727b2a466c351adc3080be9a0fea2a4c16`):
937 baris dapat didekode, 38 pengulangan exact duplicate berlabel sama dikonsolidasi,
dan dua baris dengan satu content hash tetapi label healthy/pestalotiopsis berbeda
dikarantina. Tidak ada label konflik yang diputuskan otomatis. **897 sumber layak** tersisa.

| Kelas | Raw | Sesudah konsolidasi dan karantina | Train | Validation | Test |
|---|---:|---:|---:|---:|---:|
| cordana | 162 | 162 | 113 | 24 | 25 |
| healthy | 129 | 90 | 63 | 14 | 13 |
| pestalotiopsis | 173 | 172 | 120 | 26 | 26 |
| sigatoka | 473 | 473 | 331 | 71 | 71 |
| Total | 937 | 897 | 627 | 135 | 135 |

Ada dua pasangan kandidat pHash near duplicate (jarak 2), keduanya satu label, tetapi
belum dikonfirmasi sebagai sumber sama. Laporan menyimpan kandidat dan risiko overlap.
Angka ini adalah **hasil audit data**, bukan hasil klasifikasi model. Eksekusi selalu
menghitung ulang dan memverifikasi sumber, bukan memaksakan jumlah tabel ini.
Healthy hanya 13 gambar di test, sehingga analisis kelas ini memiliki support kecil.


## Perbaikan v1.2: manifest CSV dan aktivitas GPU

Error `Manifest in-memory berbeda dari file terkunci` diperbaiki dengan loader
manifest bersama dan perbandingan identitas yang eksplisit. Identitas mencakup ID,
path sumber, label, split, grup, hash piksel, serta provenance/parent bila tersedia.
Statistik EDA float seperti brightness/contrast/sharpness tidak menentukan identitas
split. SHA-256 **seluruh file CSV** tetap diverifikasi, sehingga perubahan file,
label, keanggotaan split, hash, grup, atau sumber tetap ditolak. Perubahan nyata pada
memori menghasilkan `reports/manifest_diff_<split>.json` berisi baris/kolom penyebab.
Lock baru hanya ditulis sesudah seluruh pemeriksaan lulus.

**Memulihkan run yang sudah selesai:** pertahankan `CFG['output_dir']` yang menunjuk
artefak run Anda, restart kernel, lalu Run All notebook ini. Jangan menghapus lock,
manifest, atau checkpoint. Verifikasi lock lama tetap memeriksa semua checksum;
perbaikan reader tidak mengubah bobot, transform, policy, atau split. Run training
lama dipakai kembali. Jika isi benar-benar berbeda, baca laporan diff terlebih dahulu.

`DEVICE=cuda` dengan Tesla T4 berarti CUDA sudah dipilih. Notebook sekarang
menjalankan probe tensor CUDA nyata saat startup dan mencetak `[DEVICE]` serta
`[CACHE]`. Adanya `protocol.lock.json` membuat training dilewati. Jika validasi
manifest gagal sebelum model dimuat, GPU belum mendapat pekerjaan inference.
Prediksi test yang sudah tersimpan juga dibaca dari cache. Decode/EDA, metrik,
bootstrap dan benchmark referensi tetap menggunakan CPU.

Penyimpanan CSV/pembacaan float: [pandas read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html).
Pemeriksaan runtime: [torch.cuda.is_available](https://docs.pytorch.org/docs/stable/generated/torch.cuda.is_available.html).


Validasi perbaikan: regresi lolos pada pandas 2.2.3 dan 3.1.1, termasuk CSV BananaLSD asli, pemulihan lock lama tanpa mengubah byte lock, penolakan perubahan label/grup/hash/path, dan kegagalan freeze tanpa lock parsial. Fingerprint implementasi training/inference sama dengan v1.1 agar checkpoint tetap dapat dipakai. Eksekusi CUDA aktual perlu diperiksa melalui log probe di Kaggle; host lokal tidak memiliki GPU CUDA.


## Penggunaan GPU di Kaggle — pembaruan device

Aktifkan **Settings → Accelerator → GPU**, lalu restart session dan Run All.
Gunakan `torch`/`torchvision` bawaan image GPU Kaggle; jangan menjalankan instalasi
wheel CPU dari contoh environment lokal. Pada sel konfigurasi, periksa
`Training device: cuda | Evaluation device: cuda` serta nama GPU aktif.

Training, sanity/smoke, consistency fixtures, inferensi validation/test, dan stress
inference sekarang mengikuti device yang sama secara otomatis. Decode gambar, EDA,
transform PIL/OpenCV, metrik NumPy/scikit-learn, dan benchmark referensi host tetap
memakai CPU. Utilisasi CPU pada tahap tersebut adalah perilaku yang diharapkan.
Satu GPU digunakan; pilihan T4 x2 tidak otomatis membuat training memakai dua GPU.

Penyesuaian rencana: hasil evaluasi FP32 boleh dijalankan pada CUDA; device aktual
ikut dicatat dan dikunci di protokol. Benchmark CPU tetap terpisah. Direktori output
baru `outputs/bananalsd_notebook_device_auto_v2/` mencegah penggantian protokol CPU
lama. Jika test lama sudah dilihat, pergantian device bukan holdout baru.

Pembaruan device ini diuji dengan pemeriksaan CPU dan konfigurasi simulasi CUDA.
Eksekusi CUDA sebenarnya perlu dijalankan di session Kaggle Anda; host penyusunan
notebook tidak menyediakan GPU CUDA.

## 1. Environment

Jika dependency belum tersedia, jalankan perintah instalasi di bawah pada kernel yang dipilih,
lalu restart kernel. Contoh pasangan CPU yang dipakai untuk validasi notebook:
`torch==2.9.1`, `torchvision==0.24.1`. Untuk CUDA, gunakan pasangan kompatibel dari
[installer resmi PyTorch](https://pytorch.org/get-started/locally/).
Versi environment aktual disimpan dalam `dependency-lock.txt`; CUDA tidak dipasang oleh notebook.


In [1]:
# Jalankan secara eksplisit bila diperlukan (hapus #):
# Kaggle GPU: gunakan torch/torchvision bawaan. Jangan install wheel dari index CPU.
# Pasang hanya dependency analisis yang belum tersedia:
# %pip install numpy pandas pillow pyarrow datasets huggingface_hub scikit-learn scipy
# %pip install matplotlib pyyaml tqdm imagehash opencv-python-headless psutil nbformat


In [2]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # sebelum CUDA diinisialisasi
import copy, hashlib, importlib.metadata, io, itertools, json, math, platform
import random, subprocess, sys, tempfile, threading, time, warnings
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
from dataclasses import dataclass, asdict

import cv2, imagehash, matplotlib.pyplot as plt, numpy as np, pandas as pd
import psutil, pyarrow.parquet as pq, scipy, torch, torchvision, yaml
from PIL import Image, ImageOps, ImageEnhance
from scipy.optimize import minimize_scalar
from scipy.special import softmax, logsumexp
from scipy.stats import norm
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
    precision_recall_fscore_support, f1_score, roc_auc_score, roc_curve,
    average_precision_score)
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms as T
from torchvision.transforms import functional as TF, InterpolationMode
from torchvision.models import (mobilenet_v3_small, mobilenet_v3_large,
    MobileNet_V3_Small_Weights, MobileNet_V3_Large_Weights)
from datasets import Features, ClassLabel
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm
from IPython.display import display, Markdown

print('Python', sys.version.split()[0], '| torch', torch.__version__,
      '| torchvision', torchvision.__version__, '| CUDA', torch.cuda.is_available())

print("CUDA build:", torch.version.cuda, "| GPU count:", torch.cuda.device_count())


Python 3.13.16 | torch 2.11.0+cu130 | torchvision 0.26.0+cu130 | CUDA True
CUDA build: 13.0 | GPU count: 2


## 2. Satu konfigurasi bersama

`source_backend`: `huggingface`, `parquet` lokal, atau `directory` dengan folder empat kelas.
Local parquet harus mempunyai metadata Arrow `ClassLabel`; label numerik tanpa metadata ditolak.
CSV grup opsional: `image_id,group_id`. CSV review pHash opsional:
`image_id_a,image_id_b,confirmed_same_source` (`true`/`false`). Siapkan sebelum split dibuat.
Split yang telah ada tidak diacak ulang: fingerprint sumber/config harus sama.
Eksperimen tambahan dapat memakai `RunSpec`; tidak masuk seleksi delapan run default.


In [3]:
def discover_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'dataset.md').is_file() and (p / 'Model').is_dir():
            return p.resolve()
    return Path.cwd().resolve()  # Colab: set PROJECT_ROOT ke folder persisten Anda

PROJECT_ROOT = discover_root()
IS_KAGGLE = Path('/kaggle/working').is_dir()
RUN_MODE = 'full'  # 'smoke': diagnostik saja; 'full': delapan run dan evaluasi internal
CFG = {
    'source_backend': 'huggingface',
    'dataset_id': 'Project-AgML/BananaLSD_leaf_disease_classification',
    'dataset_config': 'raw',
    'revision': '56ad86448f39a228cc2ca72ca6bcf66634a24d7d',
    'parquet_path': 'data/raw/train-00000-of-00001.parquet',
    'directory_path': 'data/raw_images',
    'group_csv': None, 'near_duplicate_review_csv': None,
    'output_dir': 'outputs/bananalsd_notebook_device_auto_v2',
    'image_size': 224, 'seed': 42, 'ratios': [0.70, 0.15, 0.15],
    'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225],
    'resize_policy': 'direct_bilinear',
    'phash_distance': 3, 'preview_per_class': 3, 'conflict_action': 'quarantine',
    'augmentation_mode': 'online', 'offline_augmented_per_class': 400,
    'augment': {
        'crop_p': 0.35, 'crop_scale': [0.85, 1.0], 'crop_ratio': [0.90, 1.10],
        'flip_p': 0.50, 'affine_p': 0.40, 'translate': 0.05,
        'shear': 8.0, 'rotation': 15.0, 'rotation_shear': 5.0,
        'contrast_p': 0.30, 'contrast': [0.90, 1.10],
        'blur_p': 0.10, 'blur_kernel': 3, 'blur_sigma': [0.1, 1.0],
        'brightness': False, 'brightness_p': 0.20, 'brightness_range': [0.90, 1.10],
        'noise': False, 'noise_p': 0.10, 'noise_sigma': [0.01, 0.02],
        'clahe': False, 'clahe_clip': 2.0, 'clahe_grid': [8, 8],
    },
    'batch_size': 32, 'num_workers': 0, 'threads': min(4, os.cpu_count() or 1),
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'require_cuda_on_kaggle': True,
    'evaluation_device': 'same_as_training',
    'head_epochs': 5, 'finetune_epochs': 25, 'unfreeze_last': 4,
    'head_lr': 1e-3, 'backbone_lr': 1e-5, 'classifier_lr': 1e-4,
    'weight_decay': 1e-4, 'clip_norm': 1.0, 'dropout': 0.2,
    'scheduler_factor': 0.5, 'scheduler_patience': 3,
    'scheduler_threshold': 0.001, 'min_lr': 1e-7,
    'stopping_patience': 7, 'stopping_delta': 0.001,
    'confirmation_seeds': [17, 2026], 'small_margin': 0.005,
    'sanity_steps': 200, 'sanity_min_accuracy': 0.95,
    'bootstrap_replicates': 2000, 'confidence_level': 0.95, 'ece_bins': 10,
    'temperature_bounds': [0.05, 20.0], 'calibration_maxiter': 100,
    'calibration_min_total': 40, 'calibration_min_per_class': 5,
    'threshold_grid': [0.50, 0.60, 0.70, 0.80, 0.90, 0.95],
    'policy_max_risk': 0.10, 'policy_max_false_reassurance': 0.05,
    'policy_min_coverage': 0.70, 'policy_min_accepted': 50,
    'policy_min_accepted_per_class': 10,
    'benchmark_warmup': 20, 'benchmark_runs': 200, 'benchmark_sessions': 3,
    'benchmark_pipeline_images': 50,
    'stress': {'brightness': [0.85, 0.70, 0.55], 'blur': [0.5, 1.0, 1.5],
               'jpeg': [80, 60, 40], 'rotation': [5, 10, 15]},
    'targets': {'macro_f1': [0.85, 0.80], 'disease_recall': [0.90, 0.80],
                'false_reassurance': [0.05, 0.10], 'selective_risk': [0.10, 0.15],
                'coverage': [0.70, 0.60]},
}
def resolve(p):
    p = Path(p)
    return p if p.is_absolute() else PROJECT_ROOT / p

assert RUN_MODE in {'smoke', 'full'}
assert CFG['dataset_config'] == 'raw', 'Published augmented tidak boleh digabung.'
assert math.isclose(sum(CFG['ratios']), 1) and all(x > 0 for x in CFG['ratios'])
assert CFG['num_workers'] == 0, 'Referensi resume eksak notebook memakai num_workers=0.'
assert CFG['augmentation_mode'] in {'online', 'offline'}
assert CFG['conflict_action'] in {'quarantine', 'error'}
assert CFG['device'] in {'cpu', 'cuda'}
OUT = resolve(CFG['output_dir'])
OUT.mkdir(parents=True, exist_ok=True)
if IS_KAGGLE and CFG['require_cuda_on_kaggle'] and CFG['device'] != 'cuda':
    raise RuntimeError(
        f'Kaggle GPU diminta tetapi CUDA tidak tersedia. torch={torch.__version__}, '
        f'CUDA build={torch.version.cuda}, available={torch.cuda.is_available()}. '
        'Aktifkan Settings > Accelerator > GPU dan restart session. '
        'Jika pernah memasang wheel CPU, pulihkan environment GPU bawaan Kaggle. '
        'Untuk menjalankan CPU secara sengaja, set require_cuda_on_kaggle=False.')
DEVICE = torch.device(CFG['device'])
EVAL_DEVICE = DEVICE
assert CFG['evaluation_device'] == 'same_as_training'
torch.hub.set_dir(str(OUT / 'cache' / 'torch'))
torch.set_num_threads(CFG['threads'])
print('Project:', PROJECT_ROOT, '\nOutput:', OUT, '\nMode:', RUN_MODE, '| Device:', DEVICE)

print('Training device:', DEVICE, '| Evaluation device:', EVAL_DEVICE)
if DEVICE.type == 'cuda': print('GPU aktif:', torch.cuda.get_device_name(DEVICE))
print('Decode/EDA: CPU | Benchmark referensi: CPU | Model FP32, AMP off')


Project: /kaggle/working 
Output: /kaggle/working/outputs/bananalsd_notebook_device_auto_v2 
Mode: full | Device: cuda
Training device: cuda | Evaluation device: cuda
GPU aktif: Tesla T4
Decode/EDA: CPU | Benchmark referensi: CPU | Model FP32, AMP off


In [4]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()

def jsonable(x):
    if isinstance(x, dict): return {str(k): jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)): return [jsonable(v) for v in x]
    if isinstance(x, (Path, torch.device)): return str(x)
    if isinstance(x, np.ndarray): return jsonable(x.tolist())
    if isinstance(x, np.generic): return jsonable(x.item())
    if isinstance(x, float) and not math.isfinite(x): return None
    return x

def fingerprint(obj):
    return hashlib.sha256(json.dumps(jsonable(obj), sort_keys=True).encode()).hexdigest()

def fingerprint_functions(objects):
    # Bytecode/konstanta, tanpa nomor eksekusi atau nama file temporer IPython.
    def encode(value):
        if hasattr(value,'co_code'):
            return {'bytecode':value.co_code.hex(),'constants':[encode(x) for x in value.co_consts],
                    'names':list(value.co_names),'variables':list(value.co_varnames),
                    'argcount':value.co_argcount,'kwonlyargcount':value.co_kwonlyargcount}
        if isinstance(value,bytes): return {'bytes':value.hex()}
        if isinstance(value,(tuple,list)): return [encode(x) for x in value]
        if isinstance(value,(set,frozenset)):
            return sorted([encode(x) for x in value],key=lambda x:json.dumps(x,sort_keys=True))
        if isinstance(value,(str,int,float,bool,type(None))): return value
        return repr(value)
    return fingerprint({f.__qualname__:encode(f.__code__) for f in objects})

def implementation_fingerprint():
    objects=[build_model,configure_stage,train_mode,make_optimizer,run_epoch,predict_frame,
             resize_image,augment_pil,load_rgb,decode_payload,pixel_hash,
             SharedTransform.__call__,ManifestDataset.__getitem__,StressTransform.__call__,
             fit_temperature,fit_policy,acceptance,selective_metrics,metric_bundle,
             quick_stats,bootstrap_indices,confidence_intervals,paired_delta,assess_interval]
    return fingerprint_functions(objects)

def training_code_fingerprint():
    return fingerprint_functions([build_model,configure_stage,train_mode,make_optimizer,run_epoch,
        train_run,make_loader,build_transforms,SharedTransform.__call__,ManifestDataset.__getitem__,
        better,class_weights,rng_state,restore_rng,resize_image,augment_pil])

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()

def save_json(path, obj, immutable=False):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    value = jsonable(obj)
    if immutable and path.exists():
        if json.loads(path.read_text()) != value:
            raise RuntimeError(f'Artefak terkunci berbeda: {path}. Gunakan output_dir versi baru.')
        return
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')
    tmp.replace(path)

def save_csv(path, frame):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def sync(device):
    if torch.device(device).type == 'cuda': torch.cuda.synchronize(device)

def environment():
    packages = sorted(f'{d.metadata["Name"]}=={d.version}' for d in importlib.metadata.distributions())
    try:
        commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT,
                                         stderr=subprocess.DEVNULL, text=True).strip()
        dirty = bool(subprocess.check_output(['git', 'status', '--porcelain'], cwd=PROJECT_ROOT, text=True))
    except (subprocess.SubprocessError, FileNotFoundError): commit, dirty = None, None
    return {'timestamp': utc_now(), 'python': sys.version, 'os': platform.platform(),
            'cpu': platform.processor() or platform.machine(), 'ram_bytes': psutil.virtual_memory().total,
            'cuda': torch.version.cuda, 'cudnn': torch.backends.cudnn.version(),
            'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            'torch': torch.__version__, 'torchvision': torchvision.__version__,
            'threads': torch.get_num_threads(), 'deterministic': torch.are_deterministic_algorithms_enabled(),
            'precision': 'fp32', 'amp': False, 'git_commit': commit, 'git_dirty': dirty, 'packages': packages}

seed_all(CFG['seed'])
ENV = environment()
save_json(OUT / 'environment.json', ENV)
(OUT / 'dependency-lock.txt').write_text('\n'.join(ENV['packages']) + '\n')
save_json(OUT / 'config.resolved.json', CFG, immutable=True)


# Identitas split memakai kontrak data; statistik EDA float tidak menentukan identitas.
MANIFEST_REQUIRED_COLUMNS = (
    'image_id', 'source_reference', 'label_name', 'label_id', 'split', 'group_id', 'content_hash')
MANIFEST_PROVENANCE_COLUMNS = (
    'origin_reference', 'source_label_id', 'parent_image_id', 'augmentation_seed')
MANIFEST_TEXT_COLUMNS = (
    'image_id', 'source_reference', 'label_name', 'split', 'group_id', 'content_hash',
    'origin_reference', 'parent_image_id', 'phash', 'mode')

def read_manifest(path):
    # Jangan biarkan pandas menafsirkan ID seperti 001 atau NA sebagai angka/NA.
    # CSV file tetap diperiksa SHA-256-nya secara utuh oleh verifier.
    kwargs = dict(keep_default_na=False, dtype={c: str for c in MANIFEST_TEXT_COLUMNS})
    # pandas <3.1 memerlukan converter round_trip untuk mempertahankan float64.
    from packaging.version import Version
    if Version(pd.__version__) < Version('3.1'):
        kwargs['float_precision'] = 'round_trip'
    frame = pd.read_csv(path, **kwargs)
    missing = set(MANIFEST_REQUIRED_COLUMNS) - set(frame.columns)
    if missing: raise ValueError(f'Manifest {path}: kolom kurang {sorted(missing)}')
    return frame

def canonical_manifest_records(frame):
    from decimal import Decimal, InvalidOperation
    if not frame.columns.is_unique: raise ValueError('Kolom manifest ganda')
    missing = set(MANIFEST_REQUIRED_COLUMNS) - set(frame.columns)
    if missing: raise ValueError(f'Kolom identitas manifest kurang: {sorted(missing)}')
    columns = list(MANIFEST_REQUIRED_COLUMNS) + [c for c in MANIFEST_PROVENANCE_COLUMNS if c in frame]
    records = []
    for row in frame[columns].to_dict('records'):
        result = {}
        for col in columns:
            value = row[col]
            absent = value is None or value is pd.NA or (isinstance(value, (float, np.floating)) and np.isnan(value))
            if absent or value == '':
                if col in {'group_id', 'parent_image_id', 'augmentation_seed'}:
                    result[col] = ''; continue
                raise ValueError(f'Manifest memiliki {col} kosong; image_id={row.get("image_id")}')
            if col in {'label_id', 'source_label_id', 'augmentation_seed'}:
                if isinstance(value, (bool, np.bool_)): raise ValueError(f'{col} tidak boleh boolean')
                try: numeric = Decimal(str(value))
                except InvalidOperation: raise ValueError(f'{col} tidak valid: {value!r}')
                if not numeric.is_finite() or numeric != numeric.to_integral_value():
                    raise ValueError(f'{col} harus integer finite: {value!r}')
                result[col] = int(numeric)
            else:
                result[col] = str(value)
        records.append(result)
    if len({r['image_id'] for r in records}) != len(records): raise ValueError('image_id manifest ganda')
    return sorted(records, key=lambda r: r['image_id'])

def assert_manifest_equal(memory, disk, split, report_dir=None):
    left, right = canonical_manifest_records(memory), canonical_manifest_records(disk)
    if left != right:
        lm, rm = {r['image_id']: r for r in left}, {r['image_id']: r for r in right}
        common = sorted(set(lm) & set(rm))
        differences = [{'image_id':i, 'fields':{k:{'memory':lm[i].get(k),'disk':rm[i].get(k)}
            for k in set(lm[i]) | set(rm[i]) if lm[i].get(k) != rm[i].get(k)}}
            for i in common if lm[i] != rm[i]]
        report = {'split':split, 'status':'identity_mismatch',
            'only_in_memory':sorted(set(lm)-set(rm)), 'only_on_disk':sorted(set(rm)-set(lm)),
            'memory_n':len(left), 'disk_n':len(right), 'changed_rows':differences[:20],
            'comparison':'required identity + available provenance; EDA floats excluded'}
        dest = Path(report_dir or OUT/'reports') / f'manifest_diff_{split}.json'
        save_json(dest, report)
        raise RuntimeError(f'Identitas manifest berbeda: {split}. Detail baris/kolom: {dest}')
    return fingerprint(left)


def runtime_device_report(stage, model=None, expected=None, probe=False):
    target = torch.device(DEVICE if expected is None else expected)
    report = {'stage':stage, 'torch':str(torch.__version__), 'cuda_build':torch.version.cuda,
              'cuda_available':torch.cuda.is_available(), 'requested_device':str(target)}
    if target.type == 'cuda':
        if not torch.cuda.is_available():
            raise RuntimeError('DEVICE=cuda tetapi CUDA tidak tersedia; restart session dan jalankan sel konfigurasi.')
        report['gpu_name'] = torch.cuda.get_device_name(target)
        if probe:
            # Operasi tensor aktual di GPU, bukan hanya pemeriksaan is_available().
            try:
                x = torch.arange(1024, device=target, dtype=torch.float32)
                value = x.square().sum()
                torch.cuda.synchronize(target)
                if not torch.isfinite(value).item(): raise RuntimeError('Probe menghasilkan NaN/Inf')
                report['cuda_probe'] = 'passed'
                report['probe_tensor_device'] = str(value.device)
                del x, value
            except Exception as e:
                raise RuntimeError(f'CUDA terdeteksi tetapi operasi GPU gagal: {e}') from e
        report['allocated_bytes'] = torch.cuda.memory_allocated(target)
    elif probe:
        report['cuda_probe'] = 'not_requested_cpu_selected'
    if model is not None:
        actual = next(model.parameters()).device
        if actual.type != target.type or (target.index is not None and actual.index != target.index):
            raise RuntimeError(f'Model berada di {actual}, device yang diminta {target}')
        report['model_device'] = str(actual)
    print('[DEVICE]', json.dumps(report), flush=True)
    return report

STARTUP_DEVICE_REPORT = runtime_device_report('startup', probe=True)
save_json(OUT/'runtime_device_report.json', STARTUP_DEVICE_REPORT)


[DEVICE] {"stage": "startup", "torch": "2.11.0+cu130", "cuda_build": "13.0", "cuda_available": true, "requested_device": "cuda", "gpu_name": "Tesla T4", "cuda_probe": "passed", "probe_tensor_device": "cuda:0", "allocated_bytes": 0}


## 3. Baca sumber dan metadata label

Parquet HF diunduh dengan revision eksplisit dan cache lokal di output; cache tidak diduplikasi.
Raw image bytes dibaca langsung agar validasi gambar rusak dapat dicatat satu per satu.
Canonical RGB PNG adalah salinan kerja; byte sumber asli tidak ditimpa.


In [5]:
EXPECTED_CLASSES = {'cordana', 'healthy', 'pestalotiopsis', 'sigatoka'}

def load_source(cfg):
    backend = cfg['source_backend']
    records = []
    if backend in {'huggingface', 'parquet'}:
        if backend == 'huggingface':
            if not cfg['revision'] or len(cfg['revision']) != 40:
                raise ValueError('Pin revision commit HF (40 karakter), bukan main.')
            path = Path(hf_hub_download(cfg['dataset_id'],
                filename='raw/train-00000-of-00001.parquet', repo_type='dataset',
                revision=cfg['revision'], cache_dir=str(OUT / 'cache' / 'hf')))
        else:
            path = resolve(cfg['parquet_path'])
        table = pq.read_table(path)
        features = Features.from_arrow_schema(table.schema)
        if not {'image', 'label'} <= set(features): raise ValueError(f'Skema tidak didukung: {features}')
        if not isinstance(features['label'], ClassLabel):
            raise ValueError('Label numerik tanpa ClassLabel tidak dapat dipetakan dengan aman.')
        names = [n.strip().lower() for n in features['label'].names]
        source_sha = sha256_file(path)
        for i, row in enumerate(table.to_pylist()):
            raw_id = int(row['label'])
            if not 0 <= raw_id < len(names): raise ValueError(f'Label sumber invalid baris {i}')
            ref = {'dataset_id': cfg['dataset_id'], 'config': 'raw', 'source_split': 'train',
                   'revision': cfg['revision'] if backend == 'huggingface' else None,
                   'parquet_sha256': source_sha, 'row_index': i}
            records.append({'image_id': f'raw_{source_sha[:12]}_{i:05d}',
                            'label_name': names[raw_id], 'source_label_id': raw_id,
                            'payload': row['image'], 'base_dir': str(path.parent),
                            'origin_reference': json.dumps(ref, sort_keys=True)})
        provenance = {'backend': backend, 'dataset_id': cfg['dataset_id'], 'config': 'raw',
                      'revision': cfg['revision'] if backend == 'huggingface' else None,
                      'parquet_sha256': source_sha, 'source_label_names': names, 'source_rows': len(records)}
    elif backend == 'directory':
        root = resolve(cfg['directory_path'])
        names = sorted(p.name.lower() for p in root.iterdir() if p.is_dir())
        if set(names) != EXPECTED_CLASSES: raise ValueError(f'Folder kelas harus tepat {EXPECTED_CLASSES}')
        for name in names:
            for path in sorted((root / name).rglob('*')):
                if path.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}: continue
                rel = path.relative_to(root).as_posix()
                records.append({'image_id': 'local_' + fingerprint(rel)[:20], 'label_name': name,
                                'source_label_id': names.index(name), 'payload': str(path),
                                'base_dir': str(root), 'origin_reference': str(path)})
        provenance = {'backend': backend, 'root': str(root), 'source_label_names': names,
                      'source_rows': len(records), 'revision': None}
    else: raise ValueError(f'Backend tidak dikenal: {backend}')
    if len(names) != 4 or set(names) != EXPECTED_CLASSES or not records:
        raise ValueError(f'Harus empat kelas BananaLSD nonkosong, ditemukan {names}')
    mapping = {name: i for i, name in enumerate(names)}
    return records, mapping, provenance

def decode_payload(payload, base_dir=None):
    if isinstance(payload, dict):
        if payload.get('bytes') is not None: payload = io.BytesIO(payload['bytes'])
        elif payload.get('path'):
            payload = Path(payload['path'])
            if not payload.is_absolute(): payload = Path(base_dir or PROJECT_ROOT) / payload
        else: raise ValueError('Image struct tidak memiliki bytes/path')
    with Image.open(payload) as im:
        im.load()
        mode, original_size = im.mode, im.size
        rgb = ImageOps.exif_transpose(im).convert('RGB').copy()
    if min(rgb.size) <= 0: raise ValueError('Dimensi kosong')
    return rgb, mode, original_size

def load_rgb(path):
    return decode_payload(path)[0]

def pixel_hash(im):
    return hashlib.sha256(np.asarray(im.size, dtype=np.int64).tobytes() + np.asarray(im).tobytes()).hexdigest()


## 4. EDA, quality control, dan jejak sumber

EDA dihitung sebelum split. Brightness/blur hanya statistik, bukan aturan penghapusan.
Kandidat near duplicate memakai **pHash 64-bit, jarak Hamming ≤3**. File review yang
belum ada tidak menghalangi baseline; risikonya dilaporkan. Konflik exact duplicate dikarantina
seluruh anggotanya secara eksplisit; tidak dipakai supervised training/evaluasi dan tetap perlu review ahli.


In [6]:
def audit_source(records, mapping, provenance):
    started = time.perf_counter(); rows, exclusions, rgb_hist = [], [], np.zeros((3, 256), dtype=np.int64)
    cache = OUT / 'cache' / 'rgb'; cache.mkdir(parents=True, exist_ok=True)
    for r in tqdm(records, desc='Decode / quality / hash'):
        try:
            im, mode, original_size = decode_payload(r['payload'], r.get('base_dir'))
            arr = np.asarray(im); gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
            h = pixel_hash(im); dest = cache / f'{h}.png'
            if dest.exists():
                if pixel_hash(load_rgb(dest)) != h: raise RuntimeError(f'Cache berubah: {dest}')
            else: im.save(dest)
            for c in range(3): rgb_hist[c] += np.bincount(arr[:, :, c].ravel(), minlength=256)
            rows.append({k: v for k, v in r.items() if k not in {'payload', 'base_dir'}} | {
                'label_id': mapping[r['label_name']], 'source_reference': str(dest),
                'content_hash': h, 'phash': str(imagehash.phash(im, hash_size=8)),
                'width': im.width, 'height': im.height, 'aspect_ratio': im.width / im.height,
                'original_width': original_size[0], 'original_height': original_size[1], 'mode': mode,
                'brightness': float(gray.mean()), 'contrast': float(gray.std()),
                'sharpness': float(cv2.Laplacian(gray, cv2.CV_64F).var()), 'group_id': ''})
        except Exception as e:
            exclusions.append({'image_id': r['image_id'], 'origin_reference': r['origin_reference'],
                               'reason': 'decode_or_integrity_error', 'error': repr(e)})
    if not rows: raise RuntimeError('Tidak ada gambar valid. Periksa sumber.')
    frame = pd.DataFrame(rows)
    conflicts = frame.groupby('content_hash').filter(lambda g: g.label_id.nunique() > 1)
    dup = frame[frame.duplicated('content_hash', keep=False)].copy()
    pairs = []
    hashes = [int(p, 16) for p in frame.phash]
    for i, j in itertools.combinations(range(len(frame)), 2):
        dist = (hashes[i] ^ hashes[j]).bit_count()
        if dist <= CFG['phash_distance'] and frame.iloc[i].content_hash != frame.iloc[j].content_hash:
            pairs.append({'image_id_a': frame.iloc[i].image_id, 'image_id_b': frame.iloc[j].image_id,
                          'hamming_distance': dist, 'same_label': bool(frame.iloc[i].label_id == frame.iloc[j].label_id)})
    save_csv(OUT / 'reports' / 'all_valid_sources.csv', frame)
    save_csv(OUT / 'reports' / 'exact_duplicates.csv', dup)
    save_csv(OUT / 'reports' / 'near_duplicate_candidates.csv', pd.DataFrame(pairs,
        columns=['image_id_a', 'image_id_b', 'hamming_distance', 'same_label']))
    save_csv(OUT / 'reports' / 'excluded_sources.csv', pd.DataFrame(exclusions))
    # Sebelum konsolidasi: setiap sumber tercatat di all_valid_sources.csv.
    conflict_ids=set(conflicts.image_id)
    save_csv(OUT / 'reports' / 'label_review_required.csv',conflicts.assign(review_status='pending_manual_review'))
    eligible=frame[~frame.image_id.isin(conflict_ids)] if CFG['conflict_action']=='quarantine' else frame
    clean = eligible.drop_duplicates('content_hash', keep='first').copy().reset_index(drop=True)
    for r in conflicts.itertuples():
        if CFG['conflict_action']=='quarantine':
            exclusions.append({'image_id':r.image_id,'origin_reference':r.origin_reference,
                'reason':'quarantined_entire_conflicting_label_group','content_hash':r.content_hash,
                'review_status':'pending_manual_review'})
    save_csv(OUT / 'reports' / 'excluded_sources.csv',pd.DataFrame(exclusions))
    report = {'source': provenance, 'original_count': len(records), 'valid_count': len(frame),
              'excluded_count': len(exclusions), 'unique_count': len(clean),
              'exact_duplicate_extra_rows': len(eligible) - len(clean),
              'all_exact_duplicate_extra_rows':len(frame)-frame.content_hash.nunique(),
              'quarantined_conflict_rows':len(conflicts) if CFG['conflict_action']=='quarantine' else 0,
              'pending_label_review_ids':conflicts.image_id.tolist(),
              'unresolved_label_conflicts':conflicts.image_id.tolist() if CFG['conflict_action']=='error' else [],
              'conflict_action':CFG['conflict_action'],
              'near_duplicate_candidate_pairs': len(pairs), 'near_duplicate_rule': '64-bit pHash <= 3; flag only',
              'counts_original_valid': frame.label_name.value_counts().to_dict(),
              'counts_clean': clean.label_name.value_counts().to_dict(),
              'plant_leaf_session_metadata': 'not_available_unless_group_csv_supplied',
              'elapsed_seconds': time.perf_counter() - started, 'rgb_histograms': rgb_hist.tolist()}
    save_json(OUT / 'reports' / 'data_quality_report.json', report)
    if len(conflicts) and CFG['conflict_action']=='error':
        raise RuntimeError('Konflik label exact duplicate: review reports/label_review_required.csv atau pilih quarantine eksplisit.')
    if len(conflicts): warnings.warn(f'{len(conflicts)} baris konflik label dikarantina; review tetap pending. Tidak ada relabel otomatis.')
    save_json(OUT / 'label_mapping.json', mapping, immutable=True)
    return clean, frame, report

def save_fig(path, fig):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout(); fig.savefig(path, dpi=160, bbox_inches='tight'); plt.close(fig)

def eda_plots(frame, report, names):
    dest = OUT / 'eda'; dest.mkdir(parents=True, exist_ok=True)
    save_csv(dest / 'image_statistics.csv', frame)
    counts = frame.label_name.value_counts().reindex(names, fill_value=0)
    save_csv(dest / 'class_distribution.csv', pd.DataFrame({'class': names, 'n': counts.values,
                                                          'proportion': counts.values / len(frame)}))
    fig, ax = plt.subplots(figsize=(7, 4)); counts.plot.bar(ax=ax, rot=15); ax.set_ylabel('Sumber gambar valid')
    save_fig(dest / 'class_distribution.png', fig)
    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    for ax, col in zip(axes.ravel(), ['width', 'height', 'aspect_ratio', 'brightness', 'contrast', 'sharpness']):
        ax.hist(frame[col], bins=25); ax.set_title(col)
    save_fig(dest / 'quality_distributions.png', fig)
    fig, ax = plt.subplots(figsize=(7, 4))
    for color, values in zip(['red', 'green', 'blue'], report['rgb_histograms']):
        values = np.asarray(values); ax.plot(values / values.sum(), color=color, label=color)
    ax.legend(); ax.set(xlabel='Intensitas RGB', ylabel='Proporsi pixel')
    save_fig(dest / 'rgb_histogram.png', fig)
    fig, axes = plt.subplots(4, CFG['preview_per_class'], figsize=(3 * CFG['preview_per_class'], 10), squeeze=False)
    for ci, name in enumerate(names):
        part = frame[frame.label_name == name].sample(min(CFG['preview_per_class'], sum(frame.label_name == name)), random_state=42)
        for ax, (_, row) in zip(axes[ci], part.iterrows()):
            ax.imshow(load_rgb(row.source_reference)); ax.set_title(name); ax.axis('off')
    save_fig(dest / 'representative_images.png', fig)
    save_json(dest / 'summary.json', {'statistics': frame.select_dtypes('number').describe().to_dict(),
                                    'modes': frame['mode'].value_counts().to_dict()})


## 5. Split persisten dan audit kebocoran

Sel ini adalah satu-satunya pembuat split. Seed training tidak mengubah split.
Grup nonkosong, image ID, dan content hash wajib disjoint. Setiap kelas harus
tersedia di ketiga partisi. Jika grup terlalu sedikit, proses berhenti dan meminta
perbaikan upstream, bukan mengabaikan grup.


In [7]:
def assign_groups(clean, all_valid):
    clean = clean.copy(); ids = set(clean.image_id)
    canonical = clean.set_index('content_hash').image_id.to_dict()
    aliases = {r.image_id: canonical[r.content_hash] for r in all_valid.itertuples() if r.content_hash in canonical}
    parent = {x: x for x in ids}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def union(a, b):
        a, b = find(a), find(b)
        parent[max(a, b)] = min(a, b)
    grouped, reviewed = False, set()
    if CFG['group_csv']:
        meta = pd.read_csv(resolve(CFG['group_csv']), dtype=str).fillna('')
        if not {'image_id', 'group_id'} <= set(meta): raise ValueError('group_csv perlu image_id,group_id')
        if meta.image_id.duplicated().any(): raise ValueError('image_id ganda di group_csv')
        if set(meta.image_id) - set(aliases): raise ValueError('ID asing di group_csv')
        for group, rows in meta[meta.group_id != ''].groupby('group_id'):
            members = [aliases[x] for x in rows.image_id]
            for x in members[1:]: union(members[0], x)
        grouped = True
    if CFG['near_duplicate_review_csv']:
        review = pd.read_csv(resolve(CFG['near_duplicate_review_csv']), dtype=str).fillna('')
        for r in review.itertuples():
            if r.image_id_a not in aliases or r.image_id_b not in aliases: raise ValueError('ID review tidak dikenal')
            if r.confirmed_same_source.lower() not in {'true', 'false'}: raise ValueError('Review harus true/false')
            reviewed.add(tuple(sorted([r.image_id_a, r.image_id_b])))
            if r.confirmed_same_source.lower() == 'true':
                union(aliases[r.image_id_a], aliases[r.image_id_b]); grouped = True
    clean['group_id'] = [find(x) for x in clean.image_id]
    return clean, {'verified_related_groups_supplied': grouped, 'reviewed_pairs': len(reviewed),
                   'independence': 'unverified_source_capture_metadata',
                   'warning': 'Singleton group_id tidak membuktikan foto independen.'}

def check_splits(splits, mapping):
    if sorted(mapping.values()) != [0, 1, 2, 3]: raise ValueError('Mapping tidak kontigu 0..3')
    required = {'image_id', 'source_reference', 'label_name', 'label_id', 'split', 'group_id', 'content_hash'}
    for name, frame in splits.items():
        if not required <= set(frame): raise ValueError(f'Kolom manifest kurang: {name}')
        if frame.empty or frame.image_id.duplicated().any(): raise ValueError(f'ID kosong/ganda: {name}')
        if set(frame.label_id) != {0, 1, 2, 3}: raise ValueError(f'Kelas hilang: {name}')
        if not (frame.split == name).all(): raise ValueError(f'Kolom split tidak cocok: {name}')
        if any(mapping[r.label_name] != r.label_id for r in frame.itertuples()): raise ValueError('Mapping berubah')
    for a, b in itertools.combinations(splits, 2):
        for key in ['image_id', 'content_hash', 'group_id']:
            left = set(splits[a][key].dropna()) - {''}; right = set(splits[b][key].dropna()) - {''}
            if left & right: raise ValueError(f'Leakage {key}: {a}/{b}: {list(left & right)[:5]}')
    return {'status': 'passed', 'sizes': {k: len(v) for k, v in splits.items()}}

def create_splits(clean, mapping, source, group_info):
    key = fingerprint({'samples': clean[['image_id', 'content_hash', 'label_id', 'group_id']].to_dict('records'),
                       'source': source, 'ratios': CFG['ratios'], 'split_seed': 42})
    summary_path = OUT / 'reports' / 'split_summary.json'
    if summary_path.exists():
        summary = json.loads(summary_path.read_text())
        if summary['fingerprint'] != key: raise RuntimeError('Sumber/grup/split config berubah. Gunakan output_dir baru.')
        splits = {s: read_manifest(OUT / 'manifests' / f'{s}.csv') for s in ['train','validation','test']}
        for s in splits:
            if sha256_file(OUT / 'manifests' / f'{s}.csv') != summary['manifest_checksums'][s]:
                raise RuntimeError(f'Manifest berubah: {s}')
        check_splits(splits, mapping); return splits, summary
    indices = np.arange(len(clean)); y = clean.label_id.to_numpy()
    if clean.group_id.nunique() == len(clean):
        train, hold = train_test_split(indices, test_size=0.30, stratify=y, random_state=42)
        val, test = train_test_split(hold, test_size=0.50, stratify=y[hold], random_state=42)
        method = 'stratified_70_15_15'
    else:
        # Kandidat menggunakan label/grup saja; tidak memakai hasil model atau kualitas test.
        best = None
        for train, hold in GroupShuffleSplit(n_splits=500, test_size=0.30, random_state=42).split(indices, y, clean.group_id):
            for va, te in GroupShuffleSplit(n_splits=5, test_size=0.50, random_state=42).split(hold, y[hold], clean.group_id.iloc[hold]):
                parts = [train, hold[va], hold[te]]
                if any(set(y[p]) != {0,1,2,3} for p in parts): continue
                loss = sum(abs(len(p)/len(clean)-target) +
                           np.abs(np.bincount(y[p], minlength=4)/len(p)-np.bincount(y,minlength=4)/len(y)).sum()
                           for p,target in zip(parts, CFG['ratios']))
                if best is None or loss < best[0]: best = (loss, parts)
        if best is None: raise ValueError('Grup tidak memungkinkan tiga split dengan semua kelas. Review metadata upstream.')
        train, val, test = best[1]; method = 'group_disjoint_approximate_stratification'
    splits = {}
    for name, ix in zip(['train','validation','test'], [train,val,test]):
        splits[name] = clean.iloc[ix].sort_values('image_id').assign(split=name).reset_index(drop=True)
    check_splits(splits, mapping)
    checks = {}
    for name, frame in splits.items():
        path = OUT / 'manifests' / f'{name}.csv'; save_csv(path, frame); checks[name] = sha256_file(path)
    # Utilisasi berikutnya memakai representasi CSV yang sama dengan rerun.
    splits = {s: read_manifest(OUT/'manifests'/f'{s}.csv') for s in splits}
    near_path=OUT/'reports'/'near_duplicate_candidates.csv'
    near_cross=[]
    if near_path.exists():
        near=pd.read_csv(near_path)
        assignments={r.image_id:s for s,f in splits.items() for r in f.itertuples()}
        for r in near.itertuples():
            sa,sb=assignments.get(r.image_id_a),assignments.get(r.image_id_b)
            if sa and sb and sa!=sb:
                near_cross.append({'image_id_a':r.image_id_a,'split_a':sa,'image_id_b':r.image_id_b,'split_b':sb,
                                   'status':'unreviewed_candidate_risk_not_confirmed_duplicate'})
        save_csv(OUT/'reports'/'near_duplicate_split_audit.csv',pd.DataFrame(near_cross))
    summary = {'fingerprint': key, 'method': method, 'seed': 42, 'source': source,
               'group_info': group_info, 'manifest_checksums': checks,
               'unreviewed_near_duplicate_pairs_crossing_splits':near_cross,
               'sizes': {s: len(f) for s,f in splits.items()},
               'proportions': {s: len(f)/len(clean) for s,f in splits.items()},
               'class_counts': {s: f.label_name.value_counts().to_dict() for s,f in splits.items()}}
    save_json(summary_path, summary, immutable=True)
    return splits, summary


## 6. Transform bersama dan Dataset

Tujuh operasi paper diterapkan sebagai online augmentation. Translation, shear, dan
rotation+shear dipilih sebagai alternatif, bukan tiga affine kuat berturut-turut.
Reflection padding mengurangi border hitam. Besaran/probabilitas adalah default proyek,
bukan parameter yang diklaim persis dari publikasi. CLAHE/padding deterministik berlaku
konsisten pada train/validation/inference bila diaktifkan untuk eksperimen terpisah.


In [8]:
def resize_image(im, cfg):
    size = cfg['image_size']
    if cfg['resize_policy'] == 'direct_bilinear':
        return TF.resize(im, [size, size], interpolation=InterpolationMode.BILINEAR, antialias=True)
    if cfg['resize_policy'] == 'letterbox':
        return ImageOps.pad(im, (size,size), method=Image.Resampling.BILINEAR,
                            color=tuple(np.asarray(im).reshape(-1,3).mean(0).astype(int)))
    raise ValueError('resize_policy tidak dikenal')

def augment_pil(im, cfg):
    a, size = cfg['augment'], cfg['image_size']
    if random.random() < a['crop_p']:
        i,j,h,w = T.RandomResizedCrop.get_params(im, a['crop_scale'], a['crop_ratio'])
        im = TF.resized_crop(im,i,j,h,w,[size,size],InterpolationMode.BILINEAR,antialias=True)
    if random.random() < a['flip_p']: im = TF.hflip(im)
    if random.random() < a['affine_p']:
        choice = random.choice(['translation','shear','rotation_shear'])
        angle, translate, shear = 0.0, [0,0], [0.0,0.0]
        if choice == 'translation':
            translate = [round(random.uniform(-a['translate'],a['translate'])*size) for _ in range(2)]
        elif choice == 'shear': shear[0] = random.uniform(-a['shear'],a['shear'])
        else:
            angle = random.uniform(-a['rotation'],a['rotation'])
            shear[0] = random.uniform(-a['rotation_shear'],a['rotation_shear'])
        pad = math.ceil(size * 0.25)
        im = TF.pad(im, [pad,pad,pad,pad], padding_mode='reflect')
        im = TF.affine(im, angle, translate, 1.0, shear, InterpolationMode.BILINEAR)
        im = TF.center_crop(im, [size,size])
    if random.random() < a['contrast_p']: im = TF.adjust_contrast(im, random.uniform(*a['contrast']))
    if random.random() < a['blur_p']: im = TF.gaussian_blur(im,a['blur_kernel'],random.uniform(*a['blur_sigma']))
    if a['brightness'] and random.random() < a['brightness_p']:
        im = TF.adjust_brightness(im, random.uniform(*a['brightness_range']))
    return im

class SharedTransform:
    def __init__(self, cfg, split, profile='none'):
        if split not in {'train','validation','inference','test'}: raise ValueError(split)
        if profile not in {'none','paper_baseline'}: raise ValueError(profile)
        if split != 'train' and profile != 'none': raise ValueError('Augmentasi hanya untuk train')
        self.cfg, self.random = copy.deepcopy(cfg), split == 'train' and profile != 'none'
    def __call__(self, im):
        im = ImageOps.exif_transpose(im).convert('RGB')
        a = self.cfg['augment']
        if a['clahe']:
            lab = cv2.cvtColor(np.asarray(im), cv2.COLOR_RGB2LAB)
            lab[:,:,0] = cv2.createCLAHE(a['clahe_clip'], tuple(a['clahe_grid'])).apply(lab[:,:,0])
            im = Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB))
        im = resize_image(im, self.cfg)
        if self.random: im = augment_pil(im,self.cfg)
        x = TF.to_tensor(im)
        if self.random and a['noise'] and random.random() < a['noise_p']:
            x = (x + torch.randn_like(x)*random.uniform(*a['noise_sigma'])).clamp(0,1)
        if not (torch.isfinite(x).all() and x.min() >= 0 and x.max() <= 1): raise ValueError('Pixel invalid')
        return TF.normalize(x, self.cfg['mean'], self.cfg['std'])

def build_transforms(cfg, split, augmentation_profile=None):
    return SharedTransform(cfg,split,augmentation_profile or 'none')

class ManifestDataset(Dataset):
    def __init__(self, frame, transform): self.frame, self.transform = frame.reset_index(drop=True).copy(), transform
    def __len__(self): return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        try: im = load_rgb(row.source_reference); x = self.transform(im)
        except Exception as e: raise RuntimeError(f'Decode/transform gagal image_id={row.image_id}: {e}') from e
        if x.shape != (3,CFG['image_size'],CFG['image_size']) or not torch.isfinite(x).all():
            raise ValueError(f'Tensor invalid: {row.image_id}')
        return {'image':x, 'label':int(row.label_id), 'image_id':row.image_id}

def build_dataset(manifest_path, label_mapping_path, transform, source_config=None):
    frame = read_manifest(manifest_path) if not isinstance(manifest_path,pd.DataFrame) else manifest_path
    mapping = json.loads(Path(label_mapping_path).read_text()) if not isinstance(label_mapping_path,dict) else label_mapping_path
    if any(mapping[r.label_name] != r.label_id for r in frame.itertuples()): raise ValueError('Mapping dataset tidak cocok')
    return ManifestDataset(frame, transform)

def make_loader(frame, profile='none', train=False, seed=42, sampler='natural', batch_size=None):
    generator = torch.Generator().manual_seed(seed)
    ds = build_dataset(frame, MAPPING, build_transforms(CFG,'train' if train else 'inference',profile))
    sampling = None
    if train and sampler == 'weighted':
        counts = np.bincount(frame.label_id,minlength=4)
        sampling = WeightedRandomSampler(torch.as_tensor(1/counts[frame.label_id],dtype=torch.double),
                                        len(frame),replacement=True,generator=generator)
    elif sampler != 'natural': raise ValueError('Sampler tidak dikenal')
    loader = DataLoader(ds,batch_size=batch_size or CFG['batch_size'],shuffle=train and sampling is None,
        sampler=sampling,num_workers=0,pin_memory=DEVICE.type=='cuda',drop_last=False,generator=generator)
    return loader, generator

def offline_training_views(frame):
    # 400 TAMBAHAN per kelas, bukan 400 total; hanya dijalankan bila mode offline eksplisit.
    dest = OUT / 'offline_augmented'; manifest = dest / 'manifest.csv'
    key = fingerprint({'train': frame[['image_id','content_hash']].to_dict('records'), 'cfg':CFG})
    if manifest.exists():
        if json.loads((dest/'identity.json').read_text())['fingerprint'] != key: raise RuntimeError('Offline cache berbeda')
        return read_manifest(manifest)
    dest.mkdir(parents=True,exist_ok=True); rows=[]
    for c in range(4):
        subset = frame[frame.label_id==c]
        for k in range(CFG['offline_augmented_per_class']):
            seed = 42+c*100000+k; seed_all(seed)
            r = subset.iloc[k % len(subset)]; im = augment_pil(resize_image(load_rgb(r.source_reference),CFG),CFG)
            p = dest/f'{c}_{k:05d}.png'; im.save(p)
            rows.append(r.to_dict() | {'image_id':f'offline_{c}_{k:05d}', 'parent_image_id':r.image_id,
                'augmentation_seed':seed,'source_reference':str(p),'content_hash':pixel_hash(im)})
    original = frame.assign(parent_image_id='',augmentation_seed=-1)
    combined = pd.concat([original,pd.DataFrame(rows)],ignore_index=True)
    save_csv(manifest,combined); save_json(dest/'identity.json',{'fingerprint':key})
    return combined

def augmentation_preview(train):
    seed_all(42); fig,axes=plt.subplots(4,4,figsize=(10,10))
    for c in range(4):
        row=train[train.label_id==c].iloc[0]; im=resize_image(load_rgb(row.source_reference),CFG)
        for j in range(4):
            axes[c,j].imshow(im if j==0 else augment_pil(im.copy(),CFG)); axes[c,j].axis('off')
            axes[c,j].set_title(f'{row.label_name}: '+('original' if j==0 else 'train view'))
    save_fig(OUT/'augmentation_samples'/'preview.png',fig)


## 7. Jalankan preprocessing

Tidak ada bobot model dipelajari di tahap ini. Perubahan jumlah sumber akibat cleaning
akan tercatat, sehingga rasio dan support yang dilaporkan berasal dari manifest aktual.


In [9]:
if (OUT/'evaluation'/'protocol.lock.json').exists():
    print('[CACHE] Protocol sudah ada; memuat manifest terkunci. Tahap ini belum menjalankan GPU.')
    # Rerun protokol yang sama: jangan menulis ulang laporan/data yang telah di-hash.
    MAPPING=json.loads((OUT/'label_mapping.json').read_text())
    QUALITY=json.loads((OUT/'reports'/'data_quality_report.json').read_text())
    SOURCE=QUALITY['source']; ALL_VALID=pd.read_csv(OUT/'reports'/'all_valid_sources.csv',keep_default_na=False)
    SPLIT_REPORT=json.loads((OUT/'reports'/'split_summary.json').read_text())
    GROUP_INFO=SPLIT_REPORT['group_info']
    SPLITS={s:read_manifest(OUT/'manifests'/f'{s}.csv') for s in ['train','validation','test']}
    CLEAN=pd.concat(SPLITS.values(),ignore_index=True)
    raw_records=None
else:
    raw_records, MAPPING, SOURCE = load_source(CFG)
    CLEAN, ALL_VALID, QUALITY = audit_source(raw_records,MAPPING,SOURCE)
    NAMES = [name for name,_ in sorted(MAPPING.items(),key=lambda x:x[1])]
    eda_plots(ALL_VALID,QUALITY,NAMES)
    CLEAN, GROUP_INFO = assign_groups(CLEAN,ALL_VALID)
    SPLITS, SPLIT_REPORT = create_splits(CLEAN,MAPPING,SOURCE,GROUP_INFO)
    augmentation_preview(SPLITS['train'])
    save_json(OUT/'preprocessing.resolved.json',CFG)
NAMES = [name for name,_ in sorted(MAPPING.items(),key=lambda x:x[1])]
HEALTHY = MAPPING['healthy']
display(pd.DataFrame({s: f.label_name.value_counts().reindex(NAMES,fill_value=0) for s,f in SPLITS.items()}))
print('Sumber:',QUALITY['original_count'],'| valid:',len(ALL_VALID),'| unik:',len(CLEAN),
      '| excluded:',QUALITY['excluded_count'],'| kandidat near-duplicate:',QUALITY['near_duplicate_candidate_pairs'])
del raw_records  # byte sumber tidak perlu menetap di RAM selama delapan run


raw/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 28.1MB            

raw/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Decode / quality / hash:   0%|          | 0/937 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:946: UserWarning: Tag data larger than file. Skipping tag 45059
  warnings.warn(f"Tag data larger than file. Skipping tag {tag}")
/usr/local/lib/python3.13/dist-packages/PIL/JpegImagePlugin.py:898: UserWarning: Image appears to be a malformed MPO file, it will be interpreted as a base JPEG file
  warnings.warn(
/tmp/ipykernel_49/1878008814.py:66: UserWarning: 2 baris konflik label dikarantina; review tetap pending. Tidak ada relabel otomatis.
  if len(conflicts): warnings.warn(f'{len(conflicts)} baris konflik label dikarantina; review tetap pending. Tidak ada relabel otomatis.')


,train,validation,test
label_name,,,
cordana,113,24,25
healthy,63,14,13
pestalotiopsis,120,26,26
sigatoka,331,71,71


Sumber: 937 | valid: 937 | unik: 897 | excluded: 2 | kandidat near-duplicate: 2


## 8. Builder MobileNetV3 dan kebijakan freeze

Output terakhir diganti menjadi empat logit; hidden classifier dan dropout bawaan
dipertahankan (`p=0.2`). Tahap A melatih seluruh classifier. Tahap B membuka empat
**child module** terakhir `features`; semua BatchNorm backbone tetap eval dan affine frozen.


In [10]:
@dataclass(frozen=True)
class RunSpec:
    recipe: str
    variant: str = 'small'
    profile: str = 'paper_baseline'
    loss: str = 'weighted'
    sampler: str = 'natural'
    seed: int = 42
    parent_recipe: str = ''
    full_finetune: bool = False  # opsional: 10 epoch tambahan, LR 1e-6 / 1e-5

def validate_spec(spec):
    if spec.variant not in {'small','large'} or spec.profile not in {'none','paper_baseline'}: raise ValueError(spec)
    if spec.loss not in {'weighted','unweighted'} or spec.sampler not in {'natural','weighted'}: raise ValueError(spec)
    if spec.loss=='weighted' and spec.sampler=='weighted': raise ValueError('Tidak boleh double balancing')
    if CFG['augmentation_mode']=='offline' and (spec.loss=='weighted' or spec.sampler=='weighted'):
        raise ValueError('Offline balancing tidak digabung weighted loss/sampling; buat eksperimen terpisah.')

def build_model(variant='small', pretrained=True):
    if variant=='small': factory, weights=mobilenet_v3_small,MobileNet_V3_Small_Weights.IMAGENET1K_V1
    elif variant=='large': factory, weights=mobilenet_v3_large,MobileNet_V3_Large_Weights.IMAGENET1K_V2
    else: raise ValueError(variant)
    model=factory(weights=weights if pretrained else None,dropout=CFG['dropout'])
    model.classifier[-1]=nn.Linear(model.classifier[-1].in_features,4)
    assert [m.p for m in model.classifier.modules() if isinstance(m,nn.Dropout)]==[CFG['dropout']]
    model.weight_identity={'enum':f'{type(weights).__name__}.{weights.name}', 'url':weights.url}
    if pretrained:
        cached=Path(torch.hub.get_dir())/'checkpoints'/Path(weights.url).name
        model.weight_identity['file_sha256']=sha256_file(cached) if cached.exists() else None
    return model

def configure_stage(model,stage):
    for p in model.features.parameters(): p.requires_grad=False
    for p in model.classifier.parameters(): p.requires_grad=True
    opened=[]
    if stage in {'B','C'}:
        blocks=list(model.features.named_children())
        blocks=blocks[-CFG['unfreeze_last']:] if stage=='B' else blocks
        for name,block in blocks:
            opened.append(f'features.{name}')
            for p in block.parameters(): p.requires_grad=True
    for m in model.features.modules():
        if isinstance(m,nn.modules.batchnorm._BatchNorm):
            m.eval()
            for p in m.parameters(): p.requires_grad=False
    return opened

def train_mode(model):
    model.train()
    for block in model.features.children():
        if not any(p.requires_grad for p in block.parameters()): block.eval()
    for m in model.features.modules():
        if isinstance(m,nn.modules.batchnorm._BatchNorm): m.eval()

def make_optimizer(model,stage):
    groups=[]
    backbone=[p for p in model.features.parameters() if p.requires_grad]
    classifier=[p for p in model.classifier.parameters() if p.requires_grad]
    if backbone: groups.append({'params':backbone,'lr':1e-6 if stage=='C' else CFG['backbone_lr'],'name':'backbone'})
    groups.append({'params':classifier,'lr':CFG['head_lr'] if stage=='A' else (1e-5 if stage=='C' else CFG['classifier_lr']),
                   'name':'classifier'})
    flat=[id(p) for g in groups for p in g['params']]
    assert len(flat)==len(set(flat)) and set(flat)=={id(p) for p in model.parameters() if p.requires_grad}
    opt=torch.optim.AdamW(groups,weight_decay=CFG['weight_decay'])
    scheduler=None if stage=='A' else torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode='max',
        factor=CFG['scheduler_factor'],patience=CFG['scheduler_patience'],threshold=CFG['scheduler_threshold'],
        threshold_mode='abs',min_lr=CFG['min_lr'])
    return opt,scheduler

def class_weights(train):
    counts=np.bincount(train.label_id,minlength=4)
    if (counts==0).any(): raise ValueError('Kelas train kosong')
    return len(train)/(4*counts)

def basic_metrics(y,pred):
    y,pred=np.asarray(y),np.asarray(pred)
    precision,recall,f1,support=precision_recall_fscore_support(y,pred,labels=range(4),zero_division=0)
    return {'accuracy':float(np.mean(y==pred)), 'macro_f1':float(f1.mean()),
            'weighted_f1':float(np.average(f1,weights=support)),
            'balanced_accuracy':float(np.mean(recall[support>0])),
            'per_class': {str(c):{'precision':precision[c],'recall':recall[c],'f1':f1[c],'support':int(support[c])}
                          for c in range(4)}}

def better(candidate, champion):
    if champion is None: return True
    delta=candidate['macro_f1']-champion['macro_f1']
    return delta>1e-8 or (abs(delta)<=1e-8 and
        (candidate['val_ce']<champion['val_ce'] or
         (candidate['val_ce']==champion['val_ce'] and candidate['epoch']<champion['epoch'])))

def preflight():
    report=check_splits(SPLITS,MAPPING)
    if QUALITY['unresolved_label_conflicts']: raise ValueError('Konflik label belum selesai')
    active_ids=set(pd.concat(SPLITS.values()).image_id)
    if active_ids & set(QUALITY.get('pending_label_review_ids',[])):
        raise ValueError('Anggota konflik label masih berada dalam supervised split')
    for s in ['train','validation']:
        for row in SPLITS[s].itertuples():
            if pixel_hash(load_rgb(row.source_reference))!=row.content_hash: raise ValueError(f'Content berubah: {row.image_id}')
    im=load_rgb(SPLITS['validation'].iloc[0].source_reference)
    tr=build_transforms(CFG,'inference')
    assert torch.equal(tr(im),tr(im))
    assert torch.allclose(tr(Image.new('RGB',(224,224),(255,255,255)))[:,0,0],
                          (torch.ones(3)-torch.tensor(CFG['mean']))/torch.tensor(CFG['std']))
    report['test_decoded_for_training']=False
    report['source_config']='raw'; save_json(OUT/'training'/'preflight_report.json',report)
    return report


## 9. Training loop, checkpoint atomik, dan resume batas epoch

Weighted CE memakai denominator jumlah bobot target yang benar-benar teramati;
validation CE selalu biasa. Scheduler dan early stopping diperbarui sebelum `last.pt`.
Saat berpindah ke B, bobot dipulihkan dari `best_head.pt`, optimizer dibuat baru.
Checkpoint resume menyimpan RNG Python/NumPy/torch/CUDA dan generator loader.
Muat `weights_only=False` hanya untuk checkpoint training lokal yang Anda percayai.


In [11]:
def atomic_torch_save(path,obj):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp'); torch.save(obj,tmp); tmp.replace(path)

def rng_state(generator):
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),
            'cuda':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
            'loader':generator.get_state()}

def restore_rng(state,generator):
    random.setstate(state['python']); np.random.set_state(state['numpy']); torch.set_rng_state(state['torch'].cpu())
    if state['cuda'] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([x.cpu() for x in state['cuda']])
    generator.set_state(state['loader'].cpu())

def run_epoch(model,loader,optimizer=None,weights=None,max_batches=None):
    training=optimizer is not None
    train_mode(model) if training else model.eval()
    ys,preds=[],[]; numerator,denominator=0.0,0.0
    with torch.enable_grad() if training else torch.inference_mode():
        for bi,batch in enumerate(loader):
            if max_batches is not None and bi>=max_batches: break
            x=batch['image'].to(DEVICE,non_blocking=DEVICE.type=='cuda')
            y=batch['label'].to(DEVICE,dtype=torch.int64)
            if training: optimizer.zero_grad(set_to_none=True)
            logits=model(x)
            if logits.shape!=(len(y),4) or not torch.isfinite(logits).all():
                raise FloatingPointError(f'Logit invalid; image IDs: {batch.get("image_id", "fixture")}')
            losses=nn.functional.cross_entropy(logits,y,weight=weights,reduction='none')
            denom=weights[y].sum() if weights is not None else y.new_tensor(len(y),dtype=torch.float32)
            loss=losses.sum()/denom
            if not torch.isfinite(loss): raise FloatingPointError(f'Loss NaN/Inf; image IDs: {batch.get("image_id", "fixture")}')
            if training:
                loss.backward()
                for p in model.parameters():
                    if p.grad is not None and not torch.isfinite(p.grad).all():
                        raise FloatingPointError(f'Gradien NaN/Inf; image IDs: {batch.get("image_id", "fixture")}')
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],CFG['clip_norm'])
                optimizer.step()
            numerator+=float(losses.sum().detach()); denominator+=float(denom)
            ys.extend(y.detach().cpu().tolist()); preds.extend(logits.detach().argmax(1).cpu().tolist())
    if not ys: raise ValueError('Loader kosong')
    result=basic_metrics(ys,preds); result['loss']=numerator/denominator
    return result

def predict_frame(model,frame,device=None,transform=None,batch_size=None):
    device=torch.device(device or DEVICE); model.eval().to(device)
    ds=build_dataset(frame,MAPPING,transform or build_transforms(CFG,'inference'))
    loader=DataLoader(ds,batch_size=batch_size or CFG['batch_size'],shuffle=False,num_workers=0,drop_last=False)
    result=[]
    with torch.inference_mode():
        for batch in loader:
            z=model(batch['image'].to(device)).float().cpu().numpy()
            if z.shape!=(len(batch['image_id']),4) or not np.isfinite(z).all(): raise FloatingPointError('Logit invalid')
            for i,image_id in enumerate(batch['image_id']):
                result.append({'image_id':image_id,'true_label_id':int(batch['label'][i]),
                               **{f'logit_{c}':float(z[i,c]) for c in range(4)}})
    pred=pd.DataFrame(result)
    if len(pred)!=len(frame) or pred.image_id.duplicated().any() or set(pred.image_id)!=set(frame.image_id):
        raise RuntimeError('Jumlah/identitas baris prediksi tidak cocok manifest')
    return frame.merge(pred,on='image_id',how='left',validate='one_to_one')

def logits_of(frame): return frame[[f'logit_{c}' for c in range(4)]].to_numpy(float)

def model_from_checkpoint(path,variant,device=DEVICE):
    ck=torch.load(path,map_location='cpu',weights_only=False)
    model=build_model(variant,pretrained=False); model.load_state_dict(ck['model']); return model.to(device),ck

def train_run(spec,resume=False,diagnostic=False,pretrained=True,max_batches=None,epoch_limits=None):
    validate_spec(spec)
    run_id=f'{spec.recipe}_{spec.variant}_seed{spec.seed}'
    root=OUT/('diagnostics' if diagnostic else 'training')/run_id
    identity={'spec':asdict(spec),'cfg':CFG,'mapping':MAPPING,'split':SPLIT_REPORT['fingerprint'],
              'manifests':SPLIT_REPORT['manifest_checksums'],'diagnostic':diagnostic,
              'training_code_fingerprint':training_code_fingerprint(),
              'runtime':{'python':platform.python_version(),'torch':str(torch.__version__),
                         'torchvision':str(torchvision.__version__)},
              'pretrained':pretrained,'max_batches':max_batches,'epoch_limits':epoch_limits}
    key=fingerprint(identity); summary_path=root/'summary.json'
    if summary_path.exists():
        summary=json.loads(summary_path.read_text())
        if summary['fingerprint']!=key: raise RuntimeError(f'Run ID sudah dipakai config berbeda: {root}')
        if summary['status']=='completed':
            if sha256_file(summary['checkpoint'])!=summary['checkpoint_sha256']:
                raise RuntimeError(f'Checksum checkpoint cached run berubah: {run_id}')
            return summary
    if (OUT/'evaluation'/'protocol.lock.json').exists() and not diagnostic:
        raise RuntimeError('Test protocol sudah dikunci. Pengembangan baru memerlukan output/protokol versi baru.')
    last=root/'checkpoints'/'last.pt'
    if root.exists() and any(root.iterdir()) and not resume: raise RuntimeError(f'Run sudah ada; resume=True atau run ID baru: {root}')
    if resume and not last.exists(): raise FileNotFoundError(f'Checkpoint resume belum tersedia: {last}')
    root.mkdir(parents=True,exist_ok=True); save_json(root/'config.resolved.json',identity)
    save_json(root/'environment.json',environment())
    (root/'dependency-lock.txt').write_text('\n'.join(ENV['packages'])+'\n')
    seed_all(spec.seed)
    train_frame=SPLITS['train']
    online_profile=spec.profile
    if spec.profile!='none' and CFG['augmentation_mode']=='offline':
        train_frame=offline_training_views(train_frame); online_profile='none'
    train_loader,generator=make_loader(train_frame,online_profile,True,spec.seed,spec.sampler)
    val_loader,_=make_loader(SPLITS['validation'])
    weights=torch.tensor(class_weights(SPLITS['train']),dtype=torch.float32,device=DEVICE) if spec.loss=='weighted' else None
    save_json(root/'class_weights.json',{'counts_original_train':SPLITS['train'].label_id.value_counts().to_dict(),
                                      'weights':class_weights(SPLITS['train']),'applied':spec.loss=='weighted'})
    model=build_model(spec.variant,pretrained=pretrained and not resume).to(DEVICE)
    print(f'{run_id}: model parameters on {next(model.parameters()).device}', flush=True)
    champion,head_champion,history=None,None,[]
    stage,start_epoch,global_epoch='A',0,0
    stop_reference,stale=-math.inf,0; elapsed_before=0.0
    if resume:
        ck=torch.load(last,map_location='cpu',weights_only=False)
        if ck['fingerprint']!=key: raise RuntimeError('Resume menolak config/mapping/manifest berbeda')
        model.load_state_dict(ck['model']); stage=ck['stage']; start_epoch=ck['next_epoch']
        global_epoch=ck['global_epoch']; champion=ck['champion']; head_champion=ck['head_champion']
        history=ck['history']; stale=ck['stale']; stop_reference=ck['stop_reference']; elapsed_before=ck['elapsed_seconds']
    opened=configure_stage(model,stage); optimizer,scheduler=make_optimizer(model,stage)
    if resume:
        optimizer.load_state_dict(ck['optimizer'])
        # optimizer tensors harus berada pada device parameter, bukan tersisa di CPU.
        for state in optimizer.state.values():
            for k,v in state.items():
                if torch.is_tensor(v): state[k]=v.to(DEVICE)
        if scheduler is not None: scheduler.load_state_dict(ck['scheduler'])
        restore_rng(ck['rng'],generator)
    stages=['A','B']+(['C'] if spec.full_finetune else [])
    limits=epoch_limits or {'A':CFG['head_epochs'],'B':CFG['finetune_epochs'],'C':10}
    params=ck.get('parameters',{}) if resume else {}
    started=time.perf_counter(); sync(DEVICE)
    if DEVICE.type=='cuda': torch.cuda.reset_peak_memory_stats(DEVICE)
    try:
        for si in range(stages.index(stage),len(stages)):
            current=stages[si]
            if current!=stage:
                best_path=root/'checkpoints'/('best_head.pt' if current=='B' else 'best.pt')
                state=torch.load(best_path,map_location='cpu',weights_only=False)
                model.load_state_dict(state['model']); stage=current; start_epoch=0
                opened=configure_stage(model,stage); optimizer,scheduler=make_optimizer(model,stage)
                initial=run_epoch(model,val_loader,max_batches=max_batches)
                stop_reference=initial['macro_f1']; stale=0
            params[stage]={'opened_modules':opened,'total':sum(p.numel() for p in model.parameters()),
                           'trainable':sum(p.numel() for p in model.parameters() if p.requires_grad)}
            for epoch in range(start_epoch,limits[stage]):
                if stage!='A' and stale>=CFG['stopping_patience']: break
                sync(DEVICE); epoch_start=time.perf_counter()
                lrs={g['name']:g['lr'] for g in optimizer.param_groups}
                tr=run_epoch(model,train_loader,optimizer,weights,max_batches)
                va=run_epoch(model,val_loader,max_batches=max_batches)
                global_epoch+=1; sync(DEVICE)
                candidate={**va,'val_ce':va['loss'],'epoch':global_epoch,'stage':stage,'stage_epoch':epoch+1}
                row={'stage':stage,'stage_epoch':epoch+1,'epoch':global_epoch,
                     'train_objective_loss':tr['loss'],'train_accuracy':tr['accuracy'],
                     'val_ce':va['loss'],'val_accuracy':va['accuracy'],'val_macro_f1':va['macro_f1'],
                     'val_balanced_accuracy':va['balanced_accuracy'],'seconds':time.perf_counter()-epoch_start,
                     **{f'lr_{k}':v for k,v in lrs.items()},
                     **{f'val_recall_{c}':va['per_class'][str(c)]['recall'] for c in range(4)}}
                # Seluruh precision/recall/F1/support disimpan juga dalam JSON history checkpoint.
                row['per_class']=va['per_class']; history.append(row)
                improve=better(candidate,champion)
                if improve: champion=copy.deepcopy(candidate)
                head_improve=stage=='A' and better(candidate,head_champion)
                if head_improve: head_champion=copy.deepcopy(candidate)
                if scheduler is not None:
                    scheduler.step(va['macro_f1'])
                    if va['macro_f1']>=stop_reference+CFG['stopping_delta']:
                        stop_reference=va['macro_f1']; stale=0
                    else: stale+=1
                state={'model':model.state_dict(),'optimizer':optimizer.state_dict(),
                       'scheduler':scheduler.state_dict() if scheduler else None,'scaler':None,
                       'stage':stage,'next_epoch':epoch+1,'global_epoch':global_epoch,
                       'champion':champion,'head_champion':head_champion,'stale':stale,
                       'stop_reference':stop_reference,'rng':rng_state(generator),'history':history,
                       'fingerprint':key,'identity':identity,'label_mapping':MAPPING,'parameters':params,
                       'elapsed_seconds':elapsed_before+time.perf_counter()-started}
                if head_improve: atomic_torch_save(root/'checkpoints'/'best_head.pt',state)
                if improve: atomic_torch_save(root/'checkpoints'/'best.pt',state)
                atomic_torch_save(last,state)  # sesudah scheduler dan stopping state lengkap
                save_csv(root/'metrics.csv',pd.DataFrame(history).drop(columns='per_class'))
                save_json(root/'metrics.json',history)
                print(f'{run_id} {stage} {epoch+1}/{limits[stage]} | train={tr["loss"]:.4f} '
                      f'val_ce={va["loss"]:.4f} macroF1={va["macro_f1"]:.4f} | {row["seconds"]:.1f}s')
            start_epoch=0
        sync(DEVICE)
        best_model,best_state=model_from_checkpoint(root/'checkpoints'/'best.pt',spec.variant)
        validation=predict_frame(best_model,SPLITS['validation'])
        z=logits_of(validation); p=softmax(z,axis=1); validation['predicted_label_id']=p.argmax(1)
        for c in range(4): validation[f'probability_{c}']=p[:,c]
        validation['checkpoint_sha256']=sha256_file(root/'checkpoints'/'best.pt')
        validation['preprocessing_fingerprint']=fingerprint(CFG)
        save_csv(root/'validation'/'logits.csv',validation)
        save_json(root/'validation'/'metrics.json',champion)
        save_json(root/'trainable_parameters.json',params)
        summary={'run_id':run_id,'spec':asdict(spec),'status':'completed','diagnostic':diagnostic,
                 'fingerprint':key,'best':champion,'root':str(root),'checkpoint':str(root/'checkpoints'/'best.pt'),
                 'checkpoint_sha256':sha256_file(root/'checkpoints'/'best.pt'),
                 'elapsed_seconds':elapsed_before+time.perf_counter()-started,
                 'epoch_counts':dict(Counter(r['stage'] for r in history)),
                 'stage_seconds':{s:sum(r['seconds'] for r in history if r['stage']==s) for s in stages},
                 'parameters':params,'pretrained':best_model.weight_identity,
                 'peak_vram_bytes':torch.cuda.max_memory_allocated(DEVICE) if DEVICE.type=='cuda' else None}
        save_json(root/'provenance.json',{'source':SOURCE,'mapping':MAPPING,'split':SPLIT_REPORT,
                   'pretrained_weights':getattr(model,'weight_identity',{}),'run':identity})
        save_json(summary_path,summary)
        training_plots(root,history,validation)
        return summary
    except BaseException as e:
        save_json(summary_path,{'run_id':run_id,'fingerprint':key,'status':'interrupted' if isinstance(e,KeyboardInterrupt) else 'failed',
                               'error':repr(e),'diagnostic':diagnostic,'last_checkpoint':str(last) if last.exists() else None})
        raise

def confusion_plots(y,pred,root,prefix='confusion_matrix'):
    cm=confusion_matrix(y,pred,labels=range(4))
    for normalized in [False,True]:
        values=cm/np.maximum(cm.sum(1,keepdims=True),1) if normalized else cm
        fig,ax=plt.subplots(figsize=(6,5)); ax.imshow(values,cmap='Blues',vmin=0)
        ax.set(xticks=range(4),yticks=range(4),xticklabels=NAMES,yticklabels=NAMES,xlabel='Prediksi',ylabel='True label')
        plt.setp(ax.get_xticklabels(),rotation=25,ha='right')
        for i,j in itertools.product(range(4),repeat=2): ax.text(j,i,f'{values[i,j]:.2f}' if normalized else str(cm[i,j]),ha='center')
        save_fig(Path(root)/(prefix+('_normalized' if normalized else '')+'.png'),fig)

def training_plots(root,history,validation):
    h=pd.DataFrame(history); fig,axes=plt.subplots(1,3,figsize=(15,4))
    axes[0].plot(h.epoch,h.train_objective_loss,label='train objective'); axes[0].plot(h.epoch,h.val_ce,label='val CE')
    axes[1].plot(h.epoch,h.train_accuracy,label='train accuracy'); axes[1].plot(h.epoch,h.val_accuracy,label='val accuracy')
    axes[2].plot(h.epoch,h.val_macro_f1,label='validation macro-F1')
    boundary=h.loc[h.stage=='A','epoch'].max()
    for ax in axes: ax.axvline(boundary+0.5,color='gray',linestyle='--'); ax.legend(); ax.set_xlabel('Epoch')
    save_fig(Path(root)/'plots'/'learning_curves.png',fig)
    fig,ax=plt.subplots(figsize=(7,4))
    for col in h.columns:
        if col.startswith('lr_'): ax.plot(h.epoch,h[col],label=col)
    ax.set_yscale('log'); ax.legend(); save_fig(Path(root)/'plots'/'learning_rate.png',fig)
    confusion_plots(validation.true_label_id,validation.predicted_label_id,Path(root)/'validation')


## 10. Sanity, smoke, dan pengujian kontrak

Sanity memakai maksimum 16 gambar train (empat/kelas), tanpa augmentasi, model terpisah,
dan maksimum 200 update. Default gate: akurasi tiny-train ≥0.95 serta loss turun.
Ini diagnostik yang konservatif, bukan target kualitas test; kegagalan memerlukan inspeksi
label/gradien/normalisasi sebelum eksperimen penuh. Tidak ada bobot sanity dipakai ulang.


In [12]:
def sanity_check():
    root=OUT/'diagnostics'/'sanity'; root.mkdir(parents=True,exist_ok=True)
    seed_all(42); model=build_model('small',True).to(DEVICE)
    configure_stage(model,'A'); optimizer,_=make_optimizer(model,'A')
    tiny=pd.concat([SPLITS['train'][SPLITS['train'].label_id==c].head(4) for c in range(4)],ignore_index=True)
    loader,_=make_loader(tiny,train=False,batch_size=16)
    batch=next(iter(loader)); x=batch['image'].to(DEVICE); y=batch['label'].to(DEVICE)
    bn_before={n:b.clone() for n,b in model.features.named_buffers()}
    with torch.inference_mode():
        model.eval(); initial=float(nn.functional.cross_entropy(model(x),y))
    losses=[]
    for step in range(CFG['sanity_steps']):
        train_mode(model); optimizer.zero_grad(set_to_none=True); logits=model(x)
        loss=nn.functional.cross_entropy(logits,y)
        if not torch.isfinite(loss): raise FloatingPointError('Sanity loss invalid')
        loss.backward(); optimizer.step(); losses.append(float(loss.detach()))
    with torch.inference_mode():
        model.eval(); z=model(x); final=float(nn.functional.cross_entropy(z,y)); accuracy=float((z.argmax(1)==y).float().mean())
    bn_unchanged=all(torch.equal(bn_before[n],b) for n,b in model.features.named_buffers())
    passed=final<initial and accuracy>=CFG['sanity_min_accuracy'] and bn_unchanged
    report={'status':'passed' if passed else 'failed','n':len(tiny),'steps':CFG['sanity_steps'],
            'initial_loss':initial,'final_loss':final,'accuracy':accuracy,'bn_unchanged':bn_unchanged,'loss_history':losses}
    save_json(root/'report.json',report)
    if not passed: raise RuntimeError(f'Sanity gagal: lihat {root}/report.json sebelum full training')
    return report

def contract_tests():
    # Kasus risiko: overlap, kelas tidak diprediksi, tie rule, normalisasi, dan freeze.
    bad={s:f.copy() for s,f in SPLITS.items()}; bad['validation'].loc[0,'content_hash']=bad['train'].iloc[0].content_hash
    try: check_splits(bad,MAPPING)
    except ValueError: pass
    else: raise AssertionError('Leakage tidak tertangkap')
    assert basic_metrics([0,1,2,3],[0,0,0,0])['per_class']['3']['recall']==0
    champion={'macro_f1':0.8,'val_ce':0.5,'epoch':2}
    assert better({'macro_f1':0.8005,'val_ce':0.8,'epoch':3},champion)
    assert not better({'macro_f1':0.8,'val_ce':0.5,'epoch':3},champion)
    model=build_model('small',False).to(DEVICE); configure_stage(model,'A')
    before={n:p.detach().clone() for n,p in model.named_parameters()}
    buffers={n:b.detach().clone() for n,b in model.features.named_buffers()}
    opt,_=make_optimizer(model,'A'); train_mode(model)
    x=torch.randn(2,3,224,224,device=DEVICE); y=torch.tensor([0,1],device=DEVICE)
    nn.functional.cross_entropy(model(x),y).backward(); opt.step()
    assert all(torch.equal(before[n],p) for n,p in model.named_parameters() if n.startswith('features.'))
    assert any(not torch.equal(before[n],p) for n,p in model.named_parameters() if n.startswith('classifier.'))
    assert all(torch.equal(buffers[n],b) for n,b in model.features.named_buffers())
    configure_stage(model,'B'); opt,_=make_optimizer(model,'B'); train_mode(model)
    assert all(not p.requires_grad for m in model.features.modules() if isinstance(m,nn.BatchNorm2d) for p in m.parameters())
    assert sum(p.numel() for p in model.features.parameters() if p.requires_grad)>0
    return {'status':'passed','checks':['overlap','four_class_metrics','checkpoint_ties','stage_A_freeze','stage_B_groups','BN_buffers']}

if not (OUT/'evaluation'/'protocol.lock.json').exists():
    PREFLIGHT=preflight()
    CONTRACT_TESTS=contract_tests()
    SANITY=sanity_check()
    SMOKE=train_run(RunSpec('SMOKE',profile='paper_baseline',loss='weighted'),
                    diagnostic=True,pretrained=True,max_batches=2,epoch_limits={'A':1,'B':1,'C':1})
    print('Preflight/contract/sanity/smoke selesai; hasil diagnostik tidak masuk pemilihan model.')
else:
    print('Protokol sudah ada; diagnostik training tidak diulang.')


Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /kaggle/working/outputs/bananalsd_notebook_device_auto_v2/cache/torch/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 95.5MB/s]


SMOKE_small_seed42: model parameters on cuda:0
SMOKE_small_seed42 A 1/1 | train=1.5104 val_ce=1.1600 macroF1=0.3247 | 1.7s
SMOKE_small_seed42 B 1/1 | train=1.0639 val_ce=1.1095 macroF1=0.3247 | 1.4s
Preflight/contract/sanity/smoke selesai; hasil diagnostik tidak masuk pemilihan model.


## 11. Screening E00–E03, konfirmasi tiga seed, dan seleksi

E03 mengikuti resep Small terbaik, bukan menyalin bobot Small. Finalis Small dan Large
dikonfirmasi pada seed 17/2026 dengan split tetap. Jika selisih mean macro-F1 <0.005,
Small dipilih. Handoff selalu checkpoint **seed 42**, bukan seed dengan skor tertinggi.
Run yang selesai dengan fingerprint sama dipakai ulang. Untuk melanjutkan run terputus:
`train_run(RunSpec(...), resume=True)` dengan spesifikasi identik.


In [13]:
def experiment_suite():
    e00=train_run(RunSpec('E00',profile='none',loss='unweighted'))
    e01=train_run(RunSpec('E01',profile='paper_baseline',loss='unweighted'))
    e02=train_run(RunSpec('E02',profile='paper_baseline',loss='weighted'))
    screening=[e00,e01,e02]
    small=min(screening,key=lambda r:(-r['best']['macro_f1'],r['best']['val_ce'],r['spec']['recipe']))
    s=RunSpec(**small['spec'])
    large_spec=RunSpec('E03','large',s.profile,s.loss,s.sampler,42,s.recipe)
    large=train_run(large_spec); all_runs=screening+[large]
    finalists={'small':[small],'large':[large]}
    for variant,base in [('small',s),('large',large_spec)]:
        for seed in CFG['confirmation_seeds']:
            spec=RunSpec(**(asdict(base)|{'seed':seed}))
            result=train_run(spec); finalists[variant].append(result); all_runs.append(result)
    aggregates={}
    for variant,runs in finalists.items():
        aggregates[variant]={'recipe':runs[0]['spec']['recipe'],'seeds':[r['spec']['seed'] for r in runs],
            'macro_f1_mean':float(np.mean([r['best']['macro_f1'] for r in runs])),
            'macro_f1_sample_std':float(np.std([r['best']['macro_f1'] for r in runs],ddof=1)),
            'balanced_accuracy_mean':float(np.mean([r['best']['balanced_accuracy'] for r in runs])),
            'balanced_accuracy_sample_std':float(np.std([r['best']['balanced_accuracy'] for r in runs],ddof=1)),
            'recall':{str(c):{'mean':float(np.mean([r['best']['per_class'][str(c)]['recall'] for r in runs])),
                             'sample_std':float(np.std([r['best']['per_class'][str(c)]['recall'] for r in runs],ddof=1))}
                      for c in range(4)}}
    delta=aggregates['large']['macro_f1_mean']-aggregates['small']['macro_f1_mean']
    chosen='small' if abs(delta)<CFG['small_margin'] or delta<=0 else 'large'
    selected=finalists[chosen][0]
    selection={'final_recipe':selected['spec']['recipe'],'variant':chosen,'selected_seed':42,
               'selected_run':selected['run_id'],'checkpoint':selected['checkpoint'],
               'checkpoint_sha256':sha256_file(selected['checkpoint']),'best':selected['best'],
               'aggregates':aggregates,'finalists':finalists,'all_runs':all_runs,
               'reason':f'Large minus Small mean macro-F1 = {delta:.8f}; Small preference margin={CFG["small_margin"]}',
               'selection_rule':'validation mean; margin Small; fixed seed42 handoff',
               'mapping_fingerprint':fingerprint(MAPPING),'preprocessing_fingerprint':fingerprint(CFG),
               'manifest_checksums':SPLIT_REPORT['manifest_checksums'],'test_status':'not_evaluated'}
    save_json(OUT/'training'/'selection.json',selection,immutable=True)
    table=pd.DataFrame([{'run_id':r['run_id'],'recipe':r['spec']['recipe'],'variant':r['spec']['variant'],
        'seed':r['spec']['seed'],'val_macro_f1':r['best']['macro_f1'],'val_ce':r['best']['val_ce'],
        'val_balanced_accuracy':r['best']['balanced_accuracy'],'seconds':r['elapsed_seconds'],
        'stage':r['best']['stage'],'best_epoch':r['best']['epoch'],'status':r['status']} for r in all_runs])
    save_csv(OUT/'training'/'experiments.csv',table)
    majority=int(SPLITS['train'].label_id.value_counts().idxmax())
    dummy=basic_metrics(SPLITS['validation'].label_id,np.full(len(SPLITS['validation']),majority))
    save_json(OUT/'training'/'dummy_majority_validation.json',{'class_from_train':majority,'validation':dummy})
    display(table); display(pd.DataFrame(aggregates).T)
    return selection

def package_handoff(selection):
    root=OUT/'training'/'handoff'; root.mkdir(parents=True,exist_ok=True)
    model,ck=model_from_checkpoint(selection['checkpoint'],selection['variant'],device='cpu')
    if sha256_file(selection['checkpoint'])!=selection['checkpoint_sha256']: raise RuntimeError('Checkpoint berubah')
    # state_dict inference tidak mengandung optimizer/RNG.
    atomic_torch_save(root/'model_state_dict.pt',{k:v.detach().cpu() for k,v in model.state_dict().items()})
    model_cfg={'variant':selection['variant'],'num_classes':4,'dropout':CFG['dropout'],
               'weight_enum':model.weight_identity['enum'],'framework':'pytorch','precision':'fp32'}
    save_json(root/'model_config.json',model_cfg)
    save_json(root/'label_mapping.json',MAPPING)
    save_json(root/'selection.json',selection)
    save_json(root/'preprocessing.resolved.json',CFG)
    contract={'orientation':'EXIF transpose','color':'RGB','resize':CFG['resize_policy'],
              'interpolation':'bilinear_antialias','tensor_shape':['B',3,224,224],'layout':'NCHW',
              'dtype':'float32','pixel_before_normalization':[0,1],
              'normalized_range':[(0-np.array(CFG['mean']))/CFG['std'],(1-np.array(CFG['mean']))/CFG['std']],
              'mean':CFG['mean'],'std':CFG['std'],'normalization':'outside_model_once',
              'output':'four logits [B,4]','probability':'softmax(logits / T)','label_order':NAMES,
              'pretrained_transform_deviation':'direct resize; no center crop','augmentation_at_inference':False}
    save_json(root/'inference_contract.json',contract)
    save_json(root/'provenance.json',{'checkpoint_sha256':selection['checkpoint_sha256'],
        'source':SOURCE,'split':SPLIT_REPORT,'environment':ENV,'test_status':'not_evaluated'})
    source=Path(selection['checkpoint']).parent.parent/'validation'/'logits.csv'
    (root/'validation_logits.csv').write_bytes(source.read_bytes())
    (root/'test.csv').write_bytes((OUT/'manifests'/'test.csv').read_bytes())
    checks={p.name:sha256_file(p) for p in root.iterdir() if p.is_file() and p.name!='artifact_checksums.json'}
    save_json(root/'artifact_checksums.json',checks)
    return model_cfg

SELECTION=None
if RUN_MODE=='full':
    if (OUT/'evaluation'/'protocol.lock.json').exists():
        print('[CACHE] Training dilewati: protocol.lock.json sudah ada; checkpoint lama dipakai tanpa retraining.')
        SELECTION=json.loads((OUT/'training'/'selection.json').read_text())
        HANDOFF_CONFIG=json.loads((OUT/'training'/'handoff'/'model_config.json').read_text())
    else:
        SELECTION=experiment_suite()
        HANDOFF_CONFIG=package_handoff(SELECTION)
else:
    print('Mode smoke selesai. Tidak ada seleksi final, kalibrasi, atau test yang dijalankan.')


E00_small_seed42: model parameters on cuda:0
E00_small_seed42 A 1/5 | train=0.5188 val_ce=0.1993 macroF1=0.9334 | 3.7s
E00_small_seed42 A 2/5 | train=0.1310 val_ce=0.1473 macroF1=0.9364 | 3.7s
E00_small_seed42 A 3/5 | train=0.0608 val_ce=0.1166 macroF1=0.9691 | 3.7s
E00_small_seed42 A 4/5 | train=0.0299 val_ce=0.1116 macroF1=0.9590 | 3.8s
E00_small_seed42 A 5/5 | train=0.0186 val_ce=0.1117 macroF1=0.9452 | 3.8s
E00_small_seed42 B 1/25 | train=0.0301 val_ce=0.1156 macroF1=0.9628 | 4.1s
E00_small_seed42 B 2/25 | train=0.0180 val_ce=0.1122 macroF1=0.9628 | 4.0s
E00_small_seed42 B 3/25 | train=0.0126 val_ce=0.1090 macroF1=0.9628 | 4.0s
E00_small_seed42 B 4/25 | train=0.0106 val_ce=0.1092 macroF1=0.9691 | 4.0s
E00_small_seed42 B 5/25 | train=0.0076 val_ce=0.1088 macroF1=0.9628 | 3.9s
E00_small_seed42 B 6/25 | train=0.0067 val_ce=0.1072 macroF1=0.9452 | 4.0s
E00_small_seed42 B 7/25 | train=0.0037 val_ce=0.1113 macroF1=0.9691 | 3.9s
E01_small_seed42: model parameters on cuda:0
E01_small_seed4

100%|██████████| 21.1M/21.1M [00:00<00:00, 131MB/s] 


E03_large_seed42: model parameters on cuda:0
E03_large_seed42 A 1/5 | train=0.5795 val_ce=0.1601 macroF1=0.9445 | 6.8s
E03_large_seed42 A 2/5 | train=0.1066 val_ce=0.0831 macroF1=0.9623 | 6.3s
E03_large_seed42 A 3/5 | train=0.0484 val_ce=0.0831 macroF1=0.9764 | 6.7s
E03_large_seed42 A 4/5 | train=0.0182 val_ce=0.0717 macroF1=0.9654 | 6.6s
E03_large_seed42 A 5/5 | train=0.0149 val_ce=0.0712 macroF1=0.9691 | 6.5s
E03_large_seed42 B 1/25 | train=0.0204 val_ce=0.0747 macroF1=0.9827 | 7.2s
E03_large_seed42 B 2/25 | train=0.0111 val_ce=0.0714 macroF1=0.9827 | 7.2s
E03_large_seed42 B 3/25 | train=0.0080 val_ce=0.0728 macroF1=0.9758 | 6.9s
E03_large_seed42 B 4/25 | train=0.0034 val_ce=0.0804 macroF1=0.9827 | 6.9s
E03_large_seed42 B 5/25 | train=0.0116 val_ce=0.0754 macroF1=0.9827 | 6.7s
E03_large_seed42 B 6/25 | train=0.0033 val_ce=0.0740 macroF1=0.9827 | 7.2s
E03_large_seed42 B 7/25 | train=0.0029 val_ce=0.0828 macroF1=0.9827 | 7.0s
E03_large_seed42 B 8/25 | train=0.0028 val_ce=0.0777 macroF1

,run_id,recipe,variant,seed,val_macro_f1,val_ce,val_balanced_accuracy,seconds,stage,best_epoch,status
0,E00_small_seed42,E00,small,42,0.969146,0.109230,0.969551,49.016134,B,9,completed
1,E01_small_seed42,E01,small,42,0.969146,0.106391,0.969551,79.571035,B,11,completed
2,E02_small_seed42,E02,small,42,0.979410,0.107523,0.979968,124.989084,B,14,completed
3,E03_large_seed42,E03,large,42,0.982665,0.071393,0.979167,92.435216,B,7,completed
4,E02_small_seed17,E02,small,17,0.969214,0.111909,0.970353,94.418619,B,8,completed
5,E02_small_seed2026,E02,small,2026,0.979410,0.113491,0.979968,125.436196,B,13,completed
6,E03_large_seed17,E03,large,17,0.982665,0.047212,0.979167,90.596947,B,13,completed
7,E03_large_seed2026,E03,large,2026,0.982665,0.051475,0.979167,81.586740,A,5,completed


,recipe,seeds,macro_f1_mean,macro_f1_sample_std,balanced_accuracy_mean,balanced_accuracy_sample_std,recall
small,E02,"[42, 17, 2026]",0.976011,0.005886,0.976763,0.005551,"{'0': {'mean': 0.9583333333333334, 'sample_std..."
large,E03,"[42, 17, 2026]",0.982665,0.0,0.979167,0.0,"{'0': {'mean': 0.9166666666666666, 'sample_std..."


## 12. Metrik empat kelas, biner, kalibrasi, dan selective prediction

Penyakit (`D`) berarti tiga label selain healthy (`H`). False reassurance memakai
denominator **semua true D**, risiko hasil healthy memakai **accepted predicted H**.
Accepted recall kelas c memakai semua true c. Denominator nol menghasilkan `null`,
bukan nol error. Bila policy belum layak, hasil empat kelas tetap dilaporkan dan metrik
selective berstatus tidak tersedia; tidak diam-diam menerima semua gambar.


In [14]:
def proportion(num,den):
    return {'value':float(num/den) if den else None,'numerator':int(num),'denominator':int(den),
            'unit':'proportion','status':'defined' if den else 'undefined'}

def reliability(y,p):
    conf=p.max(1); correct=p.argmax(1)==y; bins=[]; ece=0.0
    edges=np.linspace(0,1,CFG['ece_bins']+1)
    for i in range(CFG['ece_bins']):
        mask=(conf>=edges[i]) & ((conf<=edges[i+1]) if i==CFG['ece_bins']-1 else (conf<edges[i+1]))
        n=int(mask.sum()); acc=float(correct[mask].mean()) if n else None; avg=float(conf[mask].mean()) if n else None
        if n: ece+=n/len(y)*abs(acc-avg)
        bins.append({'lower':edges[i],'upper':edges[i+1],'n':n,'accuracy':acc,'mean_confidence':avg})
    return {'nll':float(-np.log(p[np.arange(len(y)),y].clip(1e-15,1)).mean()),
            'brier_multiclass_sum':float(np.square(p-np.eye(4)[y]).sum(1).mean()),
            'ece_top_label':float(ece),'bins':bins}

def acceptance(p,policy):
    if not policy or policy.get('status')!='feasible': return None
    pred=p.argmax(1); thresholds=np.where(pred==HEALTHY,policy['tau_healthy'],policy['tau_disease'])
    return p.max(1)>=thresholds

def selective_metrics(y,pred,accepted):
    y,pred=np.asarray(y),np.asarray(pred); a=np.asarray(accepted,dtype=bool)
    disease=y!=HEALTHY; accepted_h=a & (pred==HEALTHY)
    result={'coverage':proportion(a.sum(),len(y)),
            'selective_risk':proportion(((pred!=y)&a).sum(),a.sum()),
            'correct_accepted_yield':proportion(((pred==y)&a).sum(),len(y)),
            'false_reassurance':proportion((disease&accepted_h).sum(),disease.sum()),
            'conditional_healthy_risk':proportion((disease&accepted_h).sum(),accepted_h.sum()),
            'accepted_predicted_healthy':int(accepted_h.sum()),
            'accepted_total':int(a.sum()),'per_class':{}}
    for c in range(4):
        mask=y==c
        result['per_class'][str(c)]={'coverage':proportion((a&mask).sum(),mask.sum()),
            'accepted_recall':proportion((a&mask&(pred==c)).sum(),mask.sum()),
            'refusal_rate':proportion((~a&mask).sum(),mask.sum()),
            'accepted_true_support':int((a&mask).sum()),'accepted_predicted_support':int((a&(pred==c)).sum())}
    result['decision_counts']={'correct_accepted':int((a&(y==pred)).sum()),
        'wrong_disease_name':int((a&(y!=HEALTHY)&(pred!=HEALTHY)&(pred!=y)).sum()),
        'disease_to_healthy':int((a&(y!=HEALTHY)&(pred==HEALTHY)).sum()),
        'healthy_to_disease':int((a&(y==HEALTHY)&(pred!=HEALTHY)).sum()),
        'uncertain':int((~a).sum()),'technical_error':0}
    return result

def metric_bundle(y,z,temperature=1.0,policy=None):
    y=np.asarray(y,dtype=int); raw=softmax(z,axis=1); p=softmax(z/temperature,axis=1); pred=p.argmax(1)
    base=basic_metrics(y,pred)
    for average in ['macro','weighted']:
        pr,re,f,_=precision_recall_fscore_support(y,pred,labels=range(4),average=average,zero_division=0)
        base.update({f'{average}_precision':float(pr),f'{average}_recall':float(re),f'{average}_f1':float(f)})
    base['confusion_matrix']=confusion_matrix(y,pred,labels=range(4)).tolist()
    base['per_class_counts']={str(c):{'correct':int(((y==c)&(pred==c)).sum()),
                                    'incorrect':int(((y==c)&(pred!=c)).sum())} for c in range(4)}
    base['ovr_roc_auc']={str(c):float(roc_auc_score(y==c,p[:,c])) if 0<(y==c).sum()<len(y) else None for c in range(4)}
    d=y!=HEALTHY; pdisease=pred!=HEALTHY; score=1-p[:,HEALTHY]
    binary={'disease_recall':proportion((d&pdisease).sum(),d.sum()),
            'healthy_specificity':proportion((~d&~pdisease).sum(),(~d).sum()),
            'disease_to_healthy':proportion((d&~pdisease).sum(),d.sum()),
            'healthy_to_disease':proportion((~d&pdisease).sum(),(~d).sum()),
            'roc_auc':float(roc_auc_score(d,score)) if 0<d.sum()<len(d) else None,
            'average_precision':float(average_precision_score(d,score)) if 0<d.sum()<len(d) else None}
    a=acceptance(p,policy)
    return {'n':len(y),'four_class':base,'binary':binary,
            'calibration_raw':reliability(y,raw),'calibration_scaled':reliability(y,p),
            'selective':selective_metrics(y,pred,a) if a is not None else {'status':'unavailable_no_feasible_policy'}}

def fit_temperature(validation,checkpoint_sha):
    if not (validation.split=='validation').all(): raise ValueError('Kalibrasi hanya pada validation, bukan test')
    z=logits_of(validation); y=validation.true_label_id.to_numpy(int)
    if set(y)!={0,1,2,3} or len(y)<CFG['calibration_min_total'] or np.bincount(y,minlength=4).min()<CFG['calibration_min_per_class']:
        return {'status':'insufficient_evidence','temperature':None,'checkpoint_sha256':checkpoint_sha}
    lo,hi=CFG['temperature_bounds']
    def nll(logt):
        scaled=z/math.exp(logt)
        return float((logsumexp(scaled,axis=1)-scaled[np.arange(len(y)),y]).mean())
    fit=minimize_scalar(nll,bounds=(math.log(lo),math.log(hi)),method='bounded',
                        options={'maxiter':CFG['calibration_maxiter'],'xatol':1e-8})
    t=float(math.exp(fit.x)) if fit.success else None
    return {'status':'fitted' if fit.success else 'optimization_failed','temperature':t,'converged':bool(fit.success),
        'iterations':int(fit.nfev),'at_bound': bool(t and (t<lo*1.01 or t>hi*0.99)),
        'nll_before':nll(0),'nll_after':nll(fit.x) if t else None,'fit_source':'validation',
        'checkpoint_sha256':checkpoint_sha,'preprocessing_fingerprint':fingerprint(CFG),
        'reuse_warning':'validation reused for model selection, calibration and policy; development optimism'}

def fit_policy(validation,temperature):
    if not (validation.split=='validation').all(): raise ValueError('Policy fitting menolak data selain validation')
    y=validation.true_label_id.to_numpy(int); p=softmax(logits_of(validation)/temperature,axis=1); pred=p.argmax(1)
    rows=[]
    for th,td in itertools.product(CFG['threshold_grid'],repeat=2):
        if th<td: continue
        policy={'status':'feasible','tau_healthy':th,'tau_disease':td}
        metrics=selective_metrics(y,pred,acceptance(p,policy))
        enough=metrics['accepted_total']>=CFG['policy_min_accepted'] and all(
            metrics['per_class'][str(c)]['accepted_true_support']>=CFG['policy_min_accepted_per_class'] for c in range(4))
        risk=metrics['selective_risk']['value']; fr=metrics['false_reassurance']['value']
        feasible=enough and risk is not None and fr is not None and risk<=CFG['policy_max_risk'] and             fr<=CFG['policy_max_false_reassurance'] and metrics['coverage']['value']>=CFG['policy_min_coverage']
        rows.append({'tau_healthy':th,'tau_disease':td,'coverage':metrics['coverage']['value'],
            'selective_risk':risk,'false_reassurance':fr,'accepted':metrics['accepted_total'],
            'accepted_predicted_healthy':metrics['accepted_predicted_healthy'],'enough_support':enough,'feasible':feasible,
            **{f'accepted_true_{c}':metrics['per_class'][str(c)]['accepted_true_support'] for c in range(4)},
            **{f'accepted_predicted_{c}':metrics['per_class'][str(c)]['accepted_predicted_support'] for c in range(4)}})
    grid=pd.DataFrame(rows); valid=grid[grid.feasible]
    if valid.empty:
        policy={'status':'no_feasible_policy' if grid.enough_support.any() else 'insufficient_evidence',
                'tau_healthy':None,'tau_disease':None,'fit_source':'validation'}
    else:
        best=valid.sort_values(['coverage','false_reassurance','tau_healthy','tau_disease'],
                               ascending=[False,True,False,False]).iloc[0]
        policy={'status':'feasible','tau_healthy':float(best.tau_healthy),'tau_disease':float(best.tau_disease),
                'fit_source':'validation','scope_detector':'not_implemented','quality_rules_enabled':False}
    return policy,grid


## 13. Interval ketidakpastian dan perbandingan berpasangan

Bootstrap 2.000 replikasi memakai grup utuh jika ada grup berulang; jika semua singleton,
dipakai bootstrap foto berstrata dan asumsi independensi foto dinyatakan eksplisit sebagai
keterbatasan. Minimal 95% replikasi harus valid. Proporsi pada singleton memakai Wilson
(bersyarat pada asumsi independensi); CI cluster degenerat dan CI macro-F1 degenerat
tidak dipakai sebagai kepastian. Variasi antar training seed dilaporkan terpisah.


In [15]:
def quick_stats(y,pred,a=None):
    cm=np.bincount(4*y+pred,minlength=16).reshape(4,4)
    support=cm.sum(1); predicted=cm.sum(0); correct=np.diag(cm)
    if (support==0).any(): macro=None; balanced=None
    else:
        macro=float(np.mean(np.divide(2*correct,support+predicted,out=np.zeros(4,dtype=float),where=support+predicted>0)))
        balanced=float(np.mean(correct/support))
    stats={'accuracy':float((y==pred).mean()),'macro_f1':macro,'balanced_accuracy':balanced}
    counts={'accuracy':(int((y==pred).sum()),len(y))}
    for c in range(4):
        counts[f'recall_{c}']=(int(correct[c]),int(support[c]))
        stats[f'recall_{c}']=float(correct[c]/support[c]) if support[c] else None
    d=y!=HEALTHY
    counts['raw_disease_to_healthy']=(int((d&(pred==HEALTHY)).sum()),int(d.sum()))
    if a is not None:
        counts.update({'coverage':(int(a.sum()),len(y)),
            'selective_risk':(int((a&(y!=pred)).sum()),int(a.sum())),
            'false_reassurance':(int((a&d&(pred==HEALTHY)).sum()),int(d.sum())),
            'conditional_healthy_risk':(int((a&d&(pred==HEALTHY)).sum()),int((a&(pred==HEALTHY)).sum())),
            'correct_accepted_yield':(int((a&(y==pred)).sum()),len(y))})
        for c in range(4):
            counts[f'accepted_recall_{c}']=(int((a&(y==c)&(pred==c)).sum()),int((y==c).sum()))
    for name,(num,den) in counts.items(): stats[name]=num/den if den else None
    return stats,counts

def bootstrap_indices(frame,rng):
    groups=frame.group_id.astype(str).to_numpy()
    # Manifest menjamin group_id nonkosong. Grup yang terkonfirmasi harus ikut bersama.
    unique=np.unique(groups)
    if len(unique)<len(frame):
        return np.concatenate([np.flatnonzero(groups==g) for g in rng.choice(unique,len(unique),replace=True)])
    y=frame.true_label_id.to_numpy(int)
    return np.concatenate([rng.choice(np.flatnonzero(y==c),sum(y==c),replace=True) for c in range(4) if (y==c).any()])

def wilson(num,den,level=0.95):
    if den==0: return None
    z=norm.ppf((1+level)/2); p=num/den; divisor=1+z*z/den
    center=(p+z*z/(2*den))/divisor
    radius=z*math.sqrt(p*(1-p)/den+z*z/(4*den*den))/divisor
    return [max(0,center-radius),min(1,center+radius)]

def confidence_intervals(frame,temperature,policy):
    y=frame.true_label_id.to_numpy(int); p=softmax(logits_of(frame)/temperature,axis=1); pred=p.argmax(1)
    a=acceptance(p,policy); point,counts=quick_stats(y,pred,a); samples={k:[] for k in point}
    rng=np.random.default_rng(CFG['seed']); reps=CFG['bootstrap_replicates']
    for _ in range(reps):
        ix=bootstrap_indices(frame,rng); values,_=quick_stats(y[ix],pred[ix],a[ix] if a is not None else None)
        for k,v in values.items():
            if v is not None and math.isfinite(v): samples[k].append(v)
    clustered=frame.group_id.nunique()<len(frame); intervals={}
    alpha=(1-CFG['confidence_level'])/2
    for k,value in point.items():
        vals=samples[k]; valid=len(vals)/reps
        ci=np.quantile(vals,[alpha,1-alpha]).tolist() if vals and valid>=0.95 else None
        method='cluster_bootstrap' if clustered else 'stratified_image_bootstrap'
        status='defined' if ci else 'insufficient_valid_replicates'
        if k in counts and not clustered and counts[k][1]>0:
            ci=wilson(*counts[k],CFG['confidence_level']); method='wilson_assuming_independent_images'; status='defined'
        elif ci is not None and abs(ci[1]-ci[0])<1e-12:
            ci=None; status='ci_not_estimable_degenerate'
        intervals[k]={'value':value,'ci':ci,'confidence_level':CFG['confidence_level'],
            'method':method,'status':status,'bootstrap_valid':len(vals),'bootstrap_total':reps,
            'valid_fraction':valid,'numerator':counts[k][0] if k in counts else None,
            'denominator':counts[k][1] if k in counts else len(y)}
    return {'unit':'verified_related_group' if clustered else 'image_assuming_independence',
            'independence_warning':GROUP_INFO['warning'],'metrics':intervals}

def paired_delta(clean,other,temperature,policy):
    other=other.set_index('image_id').loc[clean.image_id].reset_index()
    y=clean.true_label_id.to_numpy(int)
    p0=softmax(logits_of(clean)/temperature,axis=1); p1=softmax(logits_of(other)/temperature,axis=1)
    a0=acceptance(p0,policy); a1=acceptance(p1,policy)
    s0,_=quick_stats(y,p0.argmax(1),a0); s1,_=quick_stats(y,p1.argmax(1),a1)
    keep=['macro_f1','coverage','false_reassurance']+[f'recall_{c}' for c in range(4)]
    rng=np.random.default_rng(42); draws={k:[] for k in keep if k in s0}
    for _ in range(CFG['bootstrap_replicates']):
        ix=bootstrap_indices(clean,rng)
        b0,_=quick_stats(y[ix],p0.argmax(1)[ix],a0[ix] if a0 is not None else None)
        b1,_=quick_stats(y[ix],p1.argmax(1)[ix],a1[ix] if a1 is not None else None)
        for k in draws:
            if b0[k] is not None and b1[k] is not None: draws[k].append(b1[k]-b0[k])
    result={}
    for k,vals in draws.items():
        ci=np.quantile(vals,[0.025,0.975]).tolist() if len(vals)>=0.95*CFG['bootstrap_replicates'] else None
        if ci is not None and abs(ci[1]-ci[0])<1e-12: ci=None
        result[k]={'delta':s1[k]-s0[k] if s1[k] is not None and s0[k] is not None else None,
                   'ci':ci,'valid_replicates':len(vals),'method':'paired_bootstrap_same_source_groups',
                   'status':'defined' if ci else 'ci_not_estimable'}
    return result


In [16]:
def evaluation_contract_tests():
    labels=np.arange(4); y=np.tile(labels,20)
    z=np.eye(4)[y]*8
    fixture=pd.DataFrame({'image_id':[f'fixture_{i}' for i in range(len(y))],
        'group_id':[f'group_{i}' for i in range(len(y))], 'true_label_id':y,'split':'validation',
        **{f'logit_{c}':z[:,c] for c in range(4)}})
    cal=fit_temperature(fixture,'fixture_only')
    assert cal['status']=='fitted' and cal['temperature']>0
    assert np.array_equal(z.argmax(1),(z/cal['temperature']).argmax(1))
    policy,grid=fit_policy(fixture,cal['temperature'])
    assert policy['status']=='feasible' and policy['tau_healthy']>=policy['tau_disease']
    uniform=fixture.copy()
    for c in range(4): uniform[f'logit_{c}']=0.0
    assert fit_policy(uniform,1.0)[0]['status']!='feasible'
    assert fit_policy(fixture.iloc[:4],1.0)[0]['status']=='insufficient_evidence'
    for function,args in [(fit_temperature,(fixture.assign(split='test'),'fixture')),
                          (fit_policy,(fixture.assign(split='test'),1.0))]:
        try: function(*args)
        except ValueError: pass
        else: raise AssertionError('Development fitting menerima test')
    disease=[c for c in range(4) if c!=HEALTHY]
    yy=np.array([disease[0],HEALTHY,disease[1],disease[2]])
    pred=np.array([HEALTHY,HEALTHY,disease[1],disease[0]])
    m=selective_metrics(yy,pred,np.array([True,True,False,True]))
    assert m['false_reassurance']['numerator']==1 and m['false_reassurance']['denominator']==3
    assert m['conditional_healthy_risk']['denominator']==2
    assert selective_metrics(yy,pred,np.zeros(4,dtype=bool))['selective_risk']['value'] is None
    assert wilson(0,20)[1]>0 and wilson(20,20)[0]<1
    return {'status':'passed','scope':'synthetic_contract_fixture_only',
            'checks':['positive_temperature','argmax_invariance','policy_support','no_feasible_policy',
                      'test_fit_rejection','false_reassurance_denominator','zero_denominator','wilson_zero_events']}

def ood_metrics(logits,temperature,policy):
    # Antarmuka opsional untuk challenge nyata; tidak menganggap OOD sebagai kelas kelima.
    p=softmax(np.asarray(logits)/temperature,axis=1); accepted=acceptance(p,policy)
    if accepted is None:
        return {'status':'unavailable_no_feasible_policy','false_acceptance':None,'n':len(p)}
    return {'status':'defined','false_acceptance':proportion(accepted.sum(),len(p)),
            'refusal_rate':proportion((~accepted).sum(),len(p)),
            'accepted_class_counts':np.bincount(p.argmax(1)[accepted],minlength=4).tolist(),
            'scope_detector':'not_implemented','warning':'confidence refusal is not a validated scope detector'}


Transform stress dan fungsi penilaian didefinisikan sebelum protocol lock agar implementasinya ikut dibekukan.


In [17]:
class StressTransform:
    def __init__(self,name,level): self.name,self.level=name,level; self.base=build_transforms(CFG,'inference')
    def __call__(self,im):
        if self.name=='brightness': im=TF.adjust_brightness(im,self.level)
        elif self.name=='blur': im=TF.gaussian_blur(im,7,self.level)
        elif self.name=='jpeg':
            b=io.BytesIO(); im.save(b,format='JPEG',quality=int(self.level)); b.seek(0)
            with Image.open(b) as compressed: im=compressed.convert('RGB').copy()
        elif self.name=='rotation':
            w,h=im.size; pad=math.ceil(max(w,h)*0.25)
            im=TF.pad(im,pad,padding_mode='reflect'); im=TF.rotate(im,float(self.level),InterpolationMode.BILINEAR)
            im=TF.center_crop(im,[h,w])
        else: raise ValueError(self.name)
        return self.base(im)

def assess_interval(entry,point_target,ci_target,direction,min_denominator=20):
    if not entry or entry.get('value') is None: return 'inconclusive'
    value=entry['value']; ci=entry.get('ci'); den=entry.get('denominator')
    enough=den is not None and den>=min_denominator
    if (direction=='min' and value<point_target) or (direction=='max' and value>point_target): return 'below_targets'
    if not enough or ci is None or entry.get('status')!='defined': return 'inconclusive'
    passed=ci[0]>=ci_target if direction=='min' else ci[1]<=ci_target
    return 'meets_targets' if passed else 'inconclusive'


## 14. Konsistensi inference dan protocol lock

Minimal 20 development fixtures ditambah JPEG portrait/landscape, EXIF rotation,
grayscale/RGBA dan input invalid. Verifikasi ulang inference, save/load, batch 1 vs batch
evaluasi dengan `atol=1e-5, rtol=1e-4`. Test inference memverifikasi checksum dan
mencatat akses. Mengubah checkpoint/config/manifest/policy sesudah lock ditolak.


In [18]:
def load_handoff_model(device=EVAL_DEVICE):
    root=OUT/'training'/'handoff'; checks=json.loads((root/'artifact_checksums.json').read_text())
    for name,expected in checks.items():
        if sha256_file(root/name)!=expected: raise RuntimeError(f'Handoff berubah: {name}')
    config=json.loads((root/'model_config.json').read_text())
    if json.loads((root/'label_mapping.json').read_text())!=MAPPING: raise ValueError('Mapping handoff berbeda')
    model=build_model(config['variant'],False)
    model.load_state_dict(torch.load(root/'model_state_dict.pt',map_location='cpu',weights_only=True))
    loaded=model.to(device).eval()
    runtime_device_report('load_handoff_model', model=loaded, expected=device)
    return loaded

def consistency_check():
    print('Consistency fixtures on:', EVAL_DEVICE)
    root=OUT/'evaluation'/'consistency'; root.mkdir(parents=True,exist_ok=True)
    frame=pd.concat([SPLITS['validation'][SPLITS['validation'].label_id==c].head(5) for c in range(4)],ignore_index=True)
    if len(frame)<20: raise ValueError('Minimal 20 development fixture untuk evaluasi referensi')
    extras=[]; base=load_rgb(frame.iloc[0].source_reference)
    variants={'portrait':base.resize((160,320)),'landscape':base.resize((320,160)),
              'grayscale':base.convert('L'),'rgba':base.convert('RGBA')}
    exif=Image.Exif(); exif[274]=6
    exif_path=root/'exif_rotated.jpg'; base.resize((160,320)).save(exif_path,exif=exif)
    assert load_rgb(exif_path).size==(320,160)
    for name,im in variants.items():
        path=root/(name+('.png' if name=='rgba' else '.jpg')); im.save(path)
        extras.append(frame.iloc[0].to_dict()|{'image_id':'fixture_'+name,'source_reference':str(path)})
    extras.append(frame.iloc[0].to_dict()|{'image_id':'fixture_exif','source_reference':str(exif_path)})
    fixtures=pd.concat([frame,pd.DataFrame(extras)],ignore_index=True)
    model=load_handoff_model(EVAL_DEVICE)
    a=predict_frame(model,fixtures,device=EVAL_DEVICE,batch_size=1)
    b=predict_frame(model,fixtures,device=EVAL_DEVICE,batch_size=CFG['batch_size'])
    c=predict_frame(load_handoff_model(EVAL_DEVICE),fixtures,device=EVAL_DEVICE,batch_size=1)
    za,zb,zc=logits_of(a),logits_of(b),logits_of(c)
    stable=np.allclose(za,zb,atol=1e-5,rtol=1e-4) and np.allclose(za,zc,atol=1e-5,rtol=1e-4)
    stable=stable and np.array_equal(za.argmax(1),zb.argmax(1)) and np.array_equal(za.argmax(1),zc.argmax(1))
    invalid=root/'invalid.jpg'; invalid.write_bytes(b'not_an_image')
    try: load_rgb(invalid)
    except Exception as e: invalid_result={'status':'technical_error','error':type(e).__name__}
    else: raise AssertionError('Invalid image tidak ditangkap')
    tensors=torch.stack([build_transforms(CFG,'inference')(load_rgb(r.source_reference)) for r in fixtures.itertuples()])
    atomic_torch_save(root/'reference_tensors.pt',tensors)
    save_csv(root/'fixture_manifest.csv',fixtures)
    save_csv(root/'reference_logits.csv',a)
    margins=np.sort(za,axis=1)[:,-1]-np.sort(za,axis=1)[:,-2]
    report={'status':'passed' if stable else 'failed','n_valid':len(fixtures),'invalid':invalid_result,
            'atol':1e-5,'rtol':1e-4,'batch_max_abs_delta':float(np.max(np.abs(za-zb))),
            'reload_max_abs_delta':float(np.max(np.abs(za-zc))),'min_top2_logit_margin':float(margins.min()),
            'device':str(EVAL_DEVICE),'tensor_fixture_sha256':sha256_file(root/'reference_tensors.pt')}
    save_json(root/'consistency_report.json',report)
    if not stable: raise RuntimeError('Konsistensi inference gagal. Jangan buka test.')
    return report

def calibration_plots(validation,temperature):
    y=validation.true_label_id.to_numpy(int); z=logits_of(validation)
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    for name,t in [('raw',1.0),('scaled',temperature)]:
        p=softmax(z/t,axis=1); r=reliability(y,p); bins=[b for b in r['bins'] if b['n']]
        axes[0].plot([b['mean_confidence'] for b in bins],[b['accuracy'] for b in bins],marker='o',label=name)
        axes[1].hist(p.max(1),bins=np.linspace(0,1,11),alpha=0.5,label=name)
    axes[0].plot([0,1],[0,1],'k--'); axes[0].set(xlabel='Mean confidence',ylabel='Accuracy')
    axes[1].set(xlabel='Top confidence',ylabel='n validation')
    for ax in axes: ax.legend()
    save_fig(OUT/'evaluation'/'calibration'/'reliability.png',fig)

def tracked_artifacts():
    paths=[OUT/'label_mapping.json',OUT/'config.resolved.json',
           OUT/'reports'/'data_quality_report.json',OUT/'reports'/'split_summary.json',
           OUT/'training'/'selection.json',OUT/'evaluation'/'calibration'/'temperature.json',
           OUT/'evaluation'/'policy'/'decision_policy.json',
           OUT/'evaluation'/'consistency'/'reference_tensors.pt']
    paths += [OUT/'manifests'/f'{s}.csv' for s in ['train','validation','test']]
    paths += [p for p in (OUT/'training'/'handoff').iterdir() if p.is_file()]
    return {str(p.relative_to(OUT)):sha256_file(p) for p in paths}

def verify_protocol(protocol):
    if protocol['inference_device'] != str(EVAL_DEVICE):
        raise RuntimeError('Device evaluasi berbeda dari protocol lock; gunakan output versi baru.')
    if fingerprint(CFG)!=protocol['config_fingerprint'] or fingerprint(MAPPING)!=protocol['mapping_fingerprint']:
        raise RuntimeError('Config/mapping berbeda dari protocol lock')
    if implementation_fingerprint()!=protocol['implementation_fingerprint']:
        raise RuntimeError('Implementasi berubah sesudah protocol lock; buat versi pengembangan baru.')
    check_splits(SPLITS,MAPPING)
    for name,expected in protocol['artifact_checksums'].items():
        if sha256_file(OUT/name)!=expected: raise RuntimeError(f'Artefak berubah sesudah lock: {name}')
    for s in ['train','validation','test']:
        if sha256_file(OUT/'manifests'/f'{s}.csv')!=SPLIT_REPORT['manifest_checksums'][s]: raise RuntimeError('Manifest berubah')
        disk=read_manifest(OUT/'manifests'/f'{s}.csv')
        identity = assert_manifest_equal(SPLITS[s], disk, s)
        expected = protocol.get('manifest_identity_fingerprints', {}).get(s)
        if expected is not None and identity != expected:
            raise RuntimeError(f'Identitas manifest berubah dari protocol lock: {s}')

def freeze_protocol(selection,temperature,policy,consistency):
    root=OUT/'evaluation'; path=root/'protocol.lock.json'
    protocol={'version':1,'scope':'internal_only','checkpoint_seed':42,
        'checkpoint_sha256':selection['checkpoint_sha256'],
        'inference_weights_sha256':sha256_file(OUT/'training'/'handoff'/'model_state_dict.pt'),
        'config_fingerprint':fingerprint(CFG),'mapping_fingerprint':fingerprint(MAPPING),
        'implementation_fingerprint':implementation_fingerprint(),
        'artifact_checksums':tracked_artifacts(),
        'manifest_identity_fingerprints':{s:fingerprint(canonical_manifest_records(f)) for s,f in SPLITS.items()},
        'temperature':temperature,'policy':policy,
        'targets':CFG['targets'],'stress':CFG['stress'],'analysis_seed':42,
        'bootstrap_replicates':CFG['bootstrap_replicates'],'inference_device':str(EVAL_DEVICE),
        'precision':'fp32','amp':False,'consistency':consistency,'environment':ENV,
        'external':{'field_test':'pending_external_data','ood_challenge':'pending_external_data'},
        'scope_detector':'not_implemented','quality_rules':'pending_external_data',
        'tie_rule':'highest coverage, lower false reassurance, lexicographically higher (tau_H,tau_D)',
        'implementation':'single_notebook_v1'}
    if path.exists():
        old=json.loads(path.read_text()); compare={k:v for k,v in old.items() if k!='locked_at'}
        if compare!=jsonable(protocol): raise RuntimeError('Protocol sudah terkunci dengan isi berbeda. Gunakan versi output baru.')
        verify_protocol(old); return old
    # Validasi SEBELUM menulis lock; kegagalan tidak meninggalkan lock parsial.
    verify_protocol(protocol)
    protocol['locked_at']=utc_now(); save_json(path,protocol,immutable=True)
    return jsonable(protocol)

def log_test_access(protocol,operation,status,error=None):
    path=OUT/'evaluation'/'test_access_log.jsonl'; path.parent.mkdir(parents=True,exist_ok=True)
    with path.open('a') as f:
        f.write(json.dumps({'timestamp':utc_now(),'operation':operation,'status':status,'error':error,
            'protocol_sha256':sha256_file(OUT/'evaluation'/'protocol.lock.json'),
            'checkpoint_sha256':protocol['checkpoint_sha256']})+'\n')

TEMPERATURE=POLICY=PROTOCOL=None
if RUN_MODE=='full':
    if (OUT/'evaluation'/'protocol.lock.json').exists():
        print('[CACHE] Memverifikasi protocol; inference GPU belum dimulai pada tahap ini.')
        PROTOCOL=json.loads((OUT/'evaluation'/'protocol.lock.json').read_text()); verify_protocol(PROTOCOL)
        TEMPERATURE=PROTOCOL['temperature']; POLICY=PROTOCOL['policy']
        T_VALUE=TEMPERATURE['temperature'] or 1.0
    else:
        EVAL_CONTRACT_TESTS=evaluation_contract_tests()
        save_json(OUT/'evaluation'/'contract_tests.json',EVAL_CONTRACT_TESTS)
        CONSISTENCY=consistency_check()
        dev_model=load_handoff_model(EVAL_DEVICE)
        VALIDATION=predict_frame(dev_model,SPLITS['validation'],device=EVAL_DEVICE)
        VALIDATION['checkpoint_sha256']=SELECTION['checkpoint_sha256']
        VALIDATION['preprocessing_fingerprint']=fingerprint(CFG)
        save_csv(OUT/'evaluation'/'validation_logits.csv',VALIDATION)
        TEMPERATURE=fit_temperature(VALIDATION,SELECTION['checkpoint_sha256'])
        T_VALUE=TEMPERATURE['temperature'] if TEMPERATURE['status']=='fitted' else 1.0
        # T=1 adalah fallback eksplisit uncalibrated, bukan hasil fitting yang direkayasa.
        POLICY,THRESHOLD_GRID=fit_policy(VALIDATION,T_VALUE)
        save_json(OUT/'evaluation'/'calibration'/'temperature.json',TEMPERATURE)
        save_csv(OUT/'evaluation'/'policy'/'threshold_search.csv',THRESHOLD_GRID)
        save_json(OUT/'evaluation'/'policy'/'decision_policy.json',POLICY)
        DEV_METRICS=metric_bundle(VALIDATION.true_label_id.to_numpy(int),logits_of(VALIDATION),T_VALUE,POLICY)
        save_json(OUT/'evaluation'/'policy'/'development_metrics.json',DEV_METRICS)
        calibration_plots(VALIDATION,T_VALUE)
        PROTOCOL=freeze_protocol(SELECTION,TEMPERATURE,POLICY,CONSISTENCY)
    print('Protocol terkunci. Temperature:',TEMPERATURE['status'],'| Policy:',POLICY['status'])


Consistency fixtures on: cuda
[DEVICE] {"stage": "load_handoff_model", "torch": "2.11.0+cu130", "cuda_build": "13.0", "cuda_available": true, "requested_device": "cuda", "gpu_name": "Tesla T4", "allocated_bytes": 85165056, "model_device": "cuda:0"}
[DEVICE] {"stage": "load_handoff_model", "torch": "2.11.0+cu130", "cuda_build": "13.0", "cuda_available": true, "requested_device": "cuda", "gpu_name": "Tesla T4", "allocated_bytes": 102172672, "model_device": "cuda:0"}
[DEVICE] {"stage": "load_handoff_model", "torch": "2.11.0+cu130", "cuda_build": "13.0", "cuda_available": true, "requested_device": "cuda", "gpu_name": "Tesla T4", "allocated_bytes": 85165056, "model_device": "cuda:0"}
Protocol terkunci. Temperature: fitted | Policy: feasible


## 15. Test internal yang terkunci, tabel kelas, dan ROC

Test bersih memakai gambar asli tanpa balancing/augmentasi, satu baris per ID. File
rusak/content berubah menghentikan evaluasi dan dicatat sebagai error teknis. Hasil
raw dan calibrated memakai logit yang sama. Kegagalan policy tidak menghalangi
laporan empat kelas. Retry infrastruktur dicatat; skor test tidak kembali ke tuning.


In [19]:
def annotate_predictions(predictions,temperature,policy,protocol,cohort='internal_test'):
    z=logits_of(predictions); raw=softmax(z,axis=1); scaled=softmax(z/temperature,axis=1)
    result=predictions.copy(); result['predicted_label_id']=scaled.argmax(1)
    result['confidence']=scaled.max(1); result['cohort']=cohort
    a=acceptance(scaled,policy)
    result['policy_status']=np.where(a,'supported_prediction','uncertain') if a is not None else 'policy_unavailable'
    result['policy_reason']=np.where(a,'confidence_pass','below_threshold') if a is not None else policy['status']
    result['checkpoint_sha256']=protocol['checkpoint_sha256']; result['preprocessing_fingerprint']=fingerprint(CFG)
    result['torch_version']=torch.__version__; result['protocol_sha256']=sha256_file(OUT/'evaluation'/'protocol.lock.json')
    for c in range(4):
        result[f'probability_raw_{c}']=raw[:,c]; result[f'probability_calibrated_{c}']=scaled[:,c]
    return result

def roc_plots(frame,temperature,root):
    y=frame.true_label_id.to_numpy(int); p=softmax(logits_of(frame)/temperature,axis=1)
    fig,ax=plt.subplots(figsize=(7,5)); any_valid=False
    for c,name in enumerate(NAMES):
        if 0<(y==c).sum()<len(y):
            fpr,tpr,_=roc_curve(y==c,p[:,c]); ax.plot(fpr,tpr,label=f'{name}, AUC={roc_auc_score(y==c,p[:,c]):.3f}'); any_valid=True
    if any_valid:
        ax.plot([0,1],[0,1],'k--'); ax.legend(); ax.set(xlabel='False positive rate',ylabel='True positive rate')
        save_fig(Path(root)/'roc_curves.png',fig)
    else: plt.close(fig)

def evaluate_internal(protocol):
    verify_protocol(protocol); root=OUT/'evaluation'/'internal'; root.mkdir(parents=True,exist_ok=True)
    pred_path=root/'predictions.csv'; cache=root/'completion.json'
    if cache.exists():
        print('[CACHE] Prediksi test sudah tersedia; forward GPU tidak diulang.')
        meta=json.loads(cache.read_text())
        if meta['protocol_sha256']!=sha256_file(OUT/'evaluation'/'protocol.lock.json') or meta['prediction_sha256']!=sha256_file(pred_path):
            raise RuntimeError('Cache test berubah/berasal dari protocol berbeda')
        return pd.read_csv(pred_path,keep_default_na=False),json.loads((root/'metrics.json').read_text()),json.loads((root/'confidence_intervals.json').read_text())
    log_test_access(protocol,'internal_test','started')
    try:
        # Check label/content test pada tahap evaluasi, bukan pada training.
        for row in SPLITS['test'].itertuples():
            if pixel_hash(load_rgb(row.source_reference))!=row.content_hash: raise RuntimeError(f'Test content berubah {row.image_id}')
        model=load_handoff_model(EVAL_DEVICE); pred=predict_frame(model,SPLITS['test'],device=EVAL_DEVICE)
        temperature=protocol['temperature']['temperature'] or 1.0; policy=protocol['policy']
        pred=annotate_predictions(pred,temperature,policy,protocol)
        metrics=metric_bundle(pred.true_label_id.to_numpy(int),logits_of(pred),temperature,policy)
        ci=confidence_intervals(pred,temperature,policy)
        majority=int(SPLITS['train'].label_id.value_counts().idxmax())
        metrics['dummy_majority']={'majority_class_from_train':majority,
            'metrics':basic_metrics(pred.true_label_id,np.full(len(pred),majority))}
        save_csv(pred_path,pred); save_json(root/'metrics.json',metrics); save_json(root/'confidence_intervals.json',ci)
        table=pd.DataFrame(metrics['four_class']['per_class']).T; table['class_name']=NAMES
        save_csv(root/'classification_report.csv',table.reset_index(names='label_id'))
        confusion_plots(pred.true_label_id,pred.predicted_label_id,root)
        roc_plots(pred,temperature,root)
        save_json(cache,{'protocol_sha256':sha256_file(OUT/'evaluation'/'protocol.lock.json'),
                         'prediction_sha256':sha256_file(pred_path),'completed_at':utc_now()})
        log_test_access(protocol,'internal_test','completed')
        return pred,metrics,ci
    except BaseException as e:
        save_json(root/'execution_error.json',{'status':'technical_error','error':repr(e),'timestamp':utc_now()})
        log_test_access(protocol,'internal_test','failed',repr(e)); raise

TEST_PRED=TEST_METRICS=TEST_CI=None
if RUN_MODE=='full':
    TEST_PRED,TEST_METRICS,TEST_CI=evaluate_internal(PROTOCOL)
    display(pd.read_csv(OUT/'evaluation'/'internal'/'classification_report.csv'))
    print('Macro-F1:',TEST_METRICS['four_class']['macro_f1'],
          '| Accuracy:',TEST_METRICS['four_class']['accuracy'],'| Selective:',TEST_METRICS['selective'])


[DEVICE] {"stage": "load_handoff_model", "torch": "2.11.0+cu130", "cuda_build": "13.0", "cuda_available": true, "requested_device": "cuda", "gpu_name": "Tesla T4", "allocated_bytes": 102172672, "model_device": "cuda:0"}


,label_id,precision,recall,f1,support,class_name
0,0,1.000000,1.000000,1.000000,25.0,cordana
1,1,0.928571,1.000000,0.962963,13.0,healthy
2,2,0.961538,0.961538,0.961538,26.0,pestalotiopsis
3,3,1.000000,0.985915,0.992908,71.0,sigatoka


Macro-F1: 0.9793523064799661 | Accuracy: 0.9851851851851852 | Selective: {'coverage': {'value': 0.9925925925925926, 'numerator': 134, 'denominator': 135, 'unit': 'proportion', 'status': 'defined'}, 'selective_risk': {'value': 0.007462686567164179, 'numerator': 1, 'denominator': 134, 'unit': 'proportion', 'status': 'defined'}, 'correct_accepted_yield': {'value': 0.9851851851851852, 'numerator': 133, 'denominator': 135, 'unit': 'proportion', 'status': 'defined'}, 'false_reassurance': {'value': 0.0, 'numerator': 0, 'denominator': 122, 'unit': 'proportion', 'status': 'defined'}, 'conditional_healthy_risk': {'value': 0.0, 'numerator': 0, 'denominator': 13, 'unit': 'proportion', 'status': 'defined'}, 'accepted_predicted_healthy': 13, 'accepted_total': 134, 'per_class': {'0': {'coverage': {'value': 1.0, 'numerator': 25, 'denominator': 25, 'unit': 'proportion', 'status': 'defined'}, 'accepted_recall': {'value': 1.0, 'numerator': 25, 'denominator': 25, 'unit': 'proportion', 'status': 'defined'}

## 16. Stress test terpisah dan analisis error

Empat gangguan × tiga level dibekukan sebelum test; tiap turunan punya parent ID,
level, seed dan border policy. Skor clean test tidak diubah. Paired bootstrap memakai
parent/grup sama, dan melaporkan delta macro-F1/recall/coverage/false reassurance.
Gallery memprioritaskan sakit→healthy, lalu kesalahan confidence tinggi, uncertain,
serta contoh benar. Interpretasi penyebab kesalahan tetap hipotesis, tanpa mengubah label.


In [20]:
def stress_evaluation(protocol,clean):
    verify_protocol(protocol); root=OUT/'evaluation'/'stress'; root.mkdir(parents=True,exist_ok=True)
    model=load_handoff_model(EVAL_DEVICE); temperature=protocol['temperature']['temperature'] or 1.0; policy=protocol['policy']
    rows,manifest,paired=[],[],{}
    log_test_access(protocol,'stress_test','started')
    try:
        for name,levels in protocol['stress'].items():
            for level in levels:
                seed_all(42)
                pred=predict_frame(model,SPLITS['test'],device=EVAL_DEVICE,transform=StressTransform(name,level))
                key=f'{name}_{level}'
                pred=annotate_predictions(pred,temperature,policy,protocol,cohort='stress_test')
                pred['parent_image_id']=pred.image_id; pred['transform']=name; pred['level']=level; pred['seed']=42
                save_csv(root/'predictions'/f'{key}.csv',pred)
                m=metric_bundle(pred.true_label_id.to_numpy(int),logits_of(pred),temperature,policy)
                s=m['selective']; metric_row={'transform':name,'level':level,'n':len(pred),
                    'macro_f1':m['four_class']['macro_f1'],
                    'prediction_consistency':float((pred.predicted_label_id.to_numpy()==clean.predicted_label_id.to_numpy()).mean()),
                    **{f'recall_{c}':m['four_class']['per_class'][str(c)]['recall'] for c in range(4)},
                    'coverage':s.get('coverage',{}).get('value'),
                    'false_reassurance':s.get('false_reassurance',{}).get('value')}
                rows.append(metric_row); paired[key]=paired_delta(clean,pred,temperature,policy)
                for r in pred.itertuples():
                    manifest.append({'image_id':f'{r.image_id}__{key}','parent_image_id':r.image_id,
                        'group_id':r.group_id,'split':'stress_test','transform':name,'level':level,'seed':42,
                        'border':'reflection_pad_then_center_crop' if name=='rotation' else 'not_applicable'})
        save_csv(root/'manifest.csv',pd.DataFrame(manifest)); save_csv(root/'metrics.csv',pd.DataFrame(rows))
        save_json(root/'paired_deltas.json',paired); log_test_access(protocol,'stress_test','completed')
        return pd.DataFrame(rows)
    except BaseException as e:
        log_test_access(protocol,'stress_test','failed',repr(e)); raise

def error_gallery(pred):
    root=OUT/'evaluation'/'errors'; root.mkdir(parents=True,exist_ok=True)
    index=pred.copy()
    index['category']=np.select([
        (index.true_label_id!=HEALTHY)&(index.predicted_label_id==HEALTHY),
        index.true_label_id!=index.predicted_label_id,index.policy_status=='uncertain'],
        ['disease_to_healthy','other_error','uncertain'],default='correct')
    index['analysis']='not_reviewed; candidate causes require visual/expert review'
    save_csv(root/'index.csv',index)
    selected=[index[index.category=='disease_to_healthy'],
              index[index.category=='other_error'].sort_values('confidence',ascending=False),
              index[index.category=='uncertain']]
    selected += [index[(index.true_label_id==c)&(index.category=='correct')].head(3) for c in range(4)]
    gallery=pd.concat(selected).drop_duplicates('image_id')
    for start in range(0,len(gallery),12):
        part=gallery.iloc[start:start+12]; fig,axes=plt.subplots(3,4,figsize=(13,10));
        for ax in axes.ravel(): ax.axis('off')
        for ax,r in zip(axes.ravel(),part.itertuples()):
            ax.imshow(load_rgb(r.source_reference)); ax.set_title(
                f'True {NAMES[r.true_label_id]} → {NAMES[r.predicted_label_id]}\n'
                f'p={r.confidence:.3f} | {r.policy_status}\n{r.image_id}',fontsize=8)
        save_fig(root/'gallery'/f'page_{start//12+1:03d}.png',fig)
    return index

STRESS=None
if RUN_MODE=='full':
    STRESS=stress_evaluation(PROTOCOL,TEST_PRED)
    ERROR_INDEX=error_gallery(TEST_PRED)
    display(STRESS)


[DEVICE] {"stage": "load_handoff_model", "torch": "2.11.0+cu130", "cuda_build": "13.0", "cuda_available": true, "requested_device": "cuda", "gpu_name": "Tesla T4", "allocated_bytes": 102172672, "model_device": "cuda:0"}


,transform,level,n,macro_f1,prediction_consistency,recall_0,recall_1,recall_2,recall_3,coverage,false_reassurance
0,brightness,0.85,135,0.979352,1.000000,1.00,1.0,0.961538,0.985915,0.992593,0.000000
1,brightness,0.70,135,0.979352,1.000000,1.00,1.0,0.961538,0.985915,0.992593,0.000000
2,brightness,0.55,135,0.979352,1.000000,1.00,1.0,0.961538,0.985915,0.992593,0.000000
3,blur,0.50,135,0.985839,0.992593,1.00,1.0,0.961538,1.000000,0.992593,0.000000
4,blur,1.00,135,0.985839,0.992593,1.00,1.0,0.961538,1.000000,0.985185,0.000000
5,blur,1.50,135,0.917020,0.940741,0.88,1.0,0.961538,0.915493,0.962963,0.000000
6,jpeg,80.00,135,0.986863,0.992593,1.00,1.0,0.961538,0.985915,1.000000,0.000000
7,jpeg,60.00,135,0.960033,0.985185,0.92,1.0,0.961538,0.985915,0.992593,0.000000
8,jpeg,40.00,135,0.940443,0.970370,0.84,1.0,0.961538,0.985915,0.992593,0.000000
9,rotation,5.00,135,0.985839,0.992593,1.00,1.0,0.961538,1.000000,0.992593,0.000000


## 17. Efisiensi CPU host: model-only terpisah dari pipeline

Batch 1 FP32, 20 warm-up, 200 pengukuran × tiga sesi. Pipeline pada maksimal 50 foto
development memisahkan decode, EXIF/RGB, resize, tensor+normalisasi, forward,
calibration/policy, dan total. I/O merupakan pembacaan ulang cache canonical PNG;
resolusi sumber BananaLSD sudah 224, sehingga tidak mewakili decode foto kamera penuh.
Peak RSS disampling setiap 5 ms; spike lebih pendek bisa terlewat. GPU tidak diukur
oleh benchmark CPU ini dan ditulis `not_measured`. Ini bukan benchmark smartphone.


In [21]:
def timing_summary(values):
    a=np.asarray(values,dtype=float)
    return {'n':len(a),'median_ms':float(np.median(a)),'p90_ms':float(np.percentile(a,90)),
            'p95_ms':float(np.percentile(a,95)),'max_ms':float(a.max())} if len(a) else {'n':0,'status':'not_measured'}

def host_benchmark(protocol):
    print('Benchmark referensi CPU dimulai; tahap ini sengaja memakai CPU.')
    verify_protocol(protocol); root=OUT/'evaluation'/'efficiency'; root.mkdir(parents=True,exist_ok=True)
    model=load_handoff_model('cpu'); model.eval()
    frame=SPLITS['validation'].head(CFG['benchmark_pipeline_images'])
    transform=build_transforms(CFG,'inference')
    x=transform(load_rgb(frame.iloc[0].source_reference)).unsqueeze(0)
    process=psutil.Process(); baseline=process.memory_info().rss; rss=[baseline]; stop=threading.Event()
    def sample_rss():
        while not stop.wait(0.005): rss.append(process.memory_info().rss)
    sampler=threading.Thread(target=sample_rss,daemon=True); sampler.start()
    rows=[]; errors=[]
    temperature=protocol['temperature']['temperature'] or 1.0; policy=protocol['policy']
    try:
        with torch.inference_mode():
            for session in range(CFG['benchmark_sessions']):
                for _ in range(CFG['benchmark_warmup']): model(x)
                for repeat in range(CFG['benchmark_runs']):
                    start=time.perf_counter(); model(x); duration=(time.perf_counter()-start)*1000
                    rows.append({'kind':'model_only','session':session,'repeat':repeat,'component':'forward','milliseconds':duration})
            for session in range(CFG['benchmark_sessions']):
                for r in frame.itertuples():
                    begin=time.perf_counter()
                    with Image.open(r.source_reference) as original:
                        original.load(); decoded=time.perf_counter()
                        im=ImageOps.exif_transpose(original).convert('RGB'); rgb_time=time.perf_counter()
                    # Deterministic factory split into timed components, with equality check.
                    if CFG['augment']['clahe']:
                        lab=cv2.cvtColor(np.asarray(im),cv2.COLOR_RGB2LAB)
                        lab[:,:,0]=cv2.createCLAHE(CFG['augment']['clahe_clip'],tuple(CFG['augment']['clahe_grid'])).apply(lab[:,:,0])
                        im=Image.fromarray(cv2.cvtColor(lab,cv2.COLOR_LAB2RGB))
                    resized=resize_image(im,CFG); resized_time=time.perf_counter()
                    tensor=TF.normalize(TF.to_tensor(resized),CFG['mean'],CFG['std']).unsqueeze(0); tensor_time=time.perf_counter()
                    z=model(tensor).numpy(); forward_time=time.perf_counter()
                    p=softmax(z/temperature,axis=1); acceptance(p,policy); end=time.perf_counter()
                    boundaries=[begin,decoded,rgb_time,resized_time,tensor_time,forward_time,end]
                    components=['decode','exif_rgb','resize_deterministic_extras','tensor_normalize','forward','calibration_policy']
                    for component,a,b in zip(components,boundaries[:-1],boundaries[1:]):
                        rows.append({'kind':'pipeline','session':session,'image_id':r.image_id,
                                     'component':component,'milliseconds':(b-a)*1000})
                    rows.append({'kind':'pipeline','session':session,'image_id':r.image_id,'component':'total','milliseconds':(end-begin)*1000})
                    assert torch.allclose(tensor.squeeze(0),transform(load_rgb(r.source_reference)),atol=1e-6,rtol=0)
    except Exception as e:
        errors.append(repr(e)); raise
    finally:
        stop.set(); sampler.join(); rss.append(process.memory_info().rss)
        save_json(root/'memory.json',{'baseline_rss_bytes':baseline,'peak_rss_bytes':max(rss),
            'peak_above_baseline_bytes':max(rss)-baseline,'method':'process RSS sampled every 5ms',
            'limitation':'brief spikes may be missed; includes interpreter and model; not device peak',
            'gpu_peak_allocated_bytes':None,'gpu_status':'not_measured'})
    timings=pd.DataFrame(rows); save_csv(root/'raw_timings.csv',timings)
    summaries=[]
    for (kind,session,component),part in timings.groupby(['kind','session','component']):
        summaries.append({'kind':kind,'session':int(session),'component':component,**timing_summary(part.milliseconds)})
    report={'device':'cpu','precision':'fp32','batch_size':1,'warmup_per_session':CFG['benchmark_warmup'],
        'measured_per_session':CFG['benchmark_runs'],'sessions':CFG['benchmark_sessions'],
        'model_only_includes_preprocessing':False,'pipeline_io':'warm OS cache; canonical RGB PNG actual resolution',
        'parameter_count':sum(p.numel() for p in model.parameters()),
        'inference_weights_bytes':(OUT/'training'/'handoff'/'model_state_dict.pt').stat().st_size,
        'format':'PyTorch state_dict (optimizer excluded)','threads':torch.get_num_threads(),
        'environment':environment(),'summaries':summaries,'errors':errors,
        'gpu_benchmark':'not_measured','deployment_status':'not_assessed'}
    save_json(root/'host_benchmark.json',report)
    return report

HOST_BENCHMARK=None
if RUN_MODE=='full':
    HOST_BENCHMARK=host_benchmark(PROTOCOL)
    display(pd.DataFrame(HOST_BENCHMARK['summaries']))


Benchmark referensi CPU dimulai; tahap ini sengaja memakai CPU.
[DEVICE] {"stage": "load_handoff_model", "torch": "2.11.0+cu130", "cuda_build": "13.0", "cuda_available": true, "requested_device": "cpu", "model_device": "cpu"}


,kind,session,component,n,median_ms,p90_ms,p95_ms,max_ms
0,model_only,0,forward,200,26.658036,27.873757,30.121162,36.118246
1,model_only,1,forward,200,26.642758,28.103844,30.071942,80.076829
2,model_only,2,forward,200,26.697362,28.012825,30.509600,70.398078
3,pipeline,0,calibration_policy,50,0.171435,0.198833,0.203948,0.384462
4,pipeline,0,decode,50,2.478385,2.717372,2.799928,2.861868
5,pipeline,0,exif_rgb,50,0.070582,0.088263,0.095806,0.110799
6,pipeline,0,forward,50,26.937483,28.111810,29.311909,31.410492
7,pipeline,0,resize_deterministic_extras,50,0.037970,0.044422,0.046140,0.060106
8,pipeline,0,tensor_normalize,50,0.481996,0.580792,0.623233,0.698971
9,pipeline,0,total,50,30.307443,31.474929,32.520714,34.690150


## 18. Penilaian terhadap target dan laporan aktual

`completed` berarti tahap evaluasi wajib selesai, bukan model otomatis lulus target.
Aspek tanpa denominator/CI memadai menjadi `inconclusive`. Bukti eksternal tetap pending.
Laporan mencakup confusion, kelas lemah, salah sakit→healthy, dampak abstention,
robustness, biaya host, variasi seed, dan keterbatasan sumber data.


In [22]:
def write_reports(selection,metrics,ci,stress,benchmark):
    root=OUT/'evaluation'; intervals=ci['metrics']; target=CFG['targets']
    quality={}
    for name,direction in [('macro_f1','min'),('false_reassurance','max'),('selective_risk','max'),('coverage','min')]:
        quality[name]=assess_interval(intervals.get(name),*target[name],direction)
    quality['disease_recall']={NAMES[c]:assess_interval(intervals.get(f'recall_{c}'),*target['disease_recall'],'min')
                               for c in range(4) if c!=HEALTHY}
    assessment={'execution_status':'completed','evidence_scope':'internal_only',
                'quality_status':{'internal_test':quality},'targets':target,
                'external_status':{'field_test':'pending_external_data','ood_challenge':'pending_external_data',
                                   'human_quality_rules':'pending_external_data'},
                'deployment':'not_assessed','actual_metrics':metrics,'confidence_intervals':ci,
                'limitations':['small imbalanced Bangladesh dataset','capture groups incompletely known',
                               'validation reused for selection/calibration/policy','no Indonesia field/OOD evidence',
                               'confidence-only policy does not verify input scope']}
    save_json(root/'model_assessment.json',assessment)
    per=metrics['four_class']['per_class']; weak=min(range(4),key=lambda c:per[str(c)]['recall'])
    cm=np.asarray(metrics['four_class']['confusion_matrix']); off=cm.copy(); np.fill_diagonal(off,0)
    pair=np.unravel_index(np.argmax(off),off.shape)
    selective=metrics['selective']
    lines=['# Laporan evaluasi MobileNetV3 BananaLSD',
        f'Execution: completed. Evidence: internal_only. Dibuat {utc_now()}.',
        f'Checkpoint: {selection["selected_run"]}; seed 42; {selection["checkpoint_sha256"]}.',
        f'Sumber: {SOURCE}. Split aktual: {SPLIT_REPORT["sizes"]}.',
        f'Class mapping: {MAPPING}. Cleaning: {QUALITY["excluded_count"]} excluded; '
        f'{QUALITY["exact_duplicate_extra_rows"]} exact duplicate extra rows; '
        f'{QUALITY["near_duplicate_candidate_pairs"]} near-duplicate candidates.',
        '## Empat kelas',
        f'Accuracy={metrics["four_class"]["accuracy"]:.6f}; macro-F1={metrics["four_class"]["macro_f1"]:.6f}; '
        f'balanced accuracy={metrics["four_class"]["balanced_accuracy"]:.6f}.',
        f'Kelas recall terendah: {NAMES[weak]} ({per[str(weak)]["recall"]:.6f}). '
        f'Confusion terbesar: {NAMES[pair[0]]} → {NAMES[pair[1]]}: {int(off[pair])} foto.',
        f'Dummy majority dari train: {metrics["dummy_majority"]}.',
        '## Risiko biner dan keputusan',json.dumps(jsonable(metrics['binary']),indent=2),
        f'Calibration: {TEMPERATURE}. Policy: {POLICY}.',json.dumps(jsonable(selective),indent=2),
        '## Ketidakpastian dan target',json.dumps(jsonable(quality),indent=2),
        f'Bootstrap unit: {ci["unit"]}. {ci["independence_warning"]}',
        'CI lengkap dan valid replicate counts: internal/confidence_intervals.json. '
        'Meets-target bersyarat pada asumsi unit sampling yang dilaporkan; bukan validasi lapangan.',
        '## Robustness',stress.to_string(index=False),
        'Delta berpasangan: stress/paired_deltas.json; gallery: errors/gallery/. '
        'Stress merupakan karakterisasi model tetap, bukan dasar tuning ulang.',
        '## Biaya komputasi',
        f'Parameter inferensi={benchmark["parameter_count"]}; bobot={benchmark["inference_weights_bytes"]} byte; '
        f'CPU FP32 batch 1, threads={benchmark["threads"]}.',
        pd.DataFrame(benchmark['summaries']).to_string(index=False),
        'Raw timing, RSS dan keterbatasan sampling: efficiency/. Tidak menilai latency HP.',
        '## Variasi seed dan training',json.dumps(jsonable(selection['aggregates']),indent=2),
        'Kurva learning dan recall tiap epoch ada di masing-masing folder training. '
        'Periksa val CE yang meningkat saat train objective turun sebagai indikasi kemungkinan overfit; '
        'objective weighted dan CE biasa tidak setara skalanya.',
        '## Batas bukti dan tindak lanjut',
        'Field Indonesia, OOD nyata, dan quality rule dengan label manusia: pending_external_data. '
        'Kumpulkan holdout kebun/tanaman terpisah, simpan resolusi asli dan label ahli. '
        'Jika mengembangkan model sesudah melihat test ini, gunakan protokol versi baru dan holdout eksternal baru.',
        'Healthy hanya berarti tidak teridentifikasi tiga penyakit daun yang didukung pada foto; '
        'bukan jaminan seluruh tanaman sehat. Model belum membuktikan semua penyakit pisang, severity, '
        'infeksi campuran, domain Indonesia, atau kesiapan aplikasi.']
    (root/'evaluation_report.md').write_text('\n\n'.join(lines),encoding='utf-8')
    card=['# Model card',f'Architecture: MobileNetV3-{selection["variant"]}; FP32; output order {NAMES}.',
        'Intended use: penelitian klasifikasi empat kategori foto daun BananaLSD.',
        'Unsupported: diagnosis seluruh tanaman, pengobatan, semua penyakit, severity, dan domain baru tanpa validasi.',
        f'Dataset revision {SOURCE.get("revision")}; split counts {SPLIT_REPORT["sizes"]}.',
        f'Checkpoint provenance: {selection["checkpoint_sha256"]}.',
        f'Internal test macro-F1 {metrics["four_class"]["macro_f1"]:.6f}; lihat CI dan metrik per kelas pada laporan.',
        f'Calibration {TEMPERATURE["status"]}; policy {POLICY["status"]}; scope detector not_implemented.',
        'Generalization: internal_only; field/OOD pending_external_data. Tidak ada klaim deployment.']
    (root/'model_card.md').write_text('\n\n'.join(card),encoding='utf-8')
    summary=[f'Model terpilih: {selection["selected_run"]}.',
        f'Recall terendah: {NAMES[weak]}. Kasus sakit→healthy: {metrics["binary"]["disease_to_healthy"]}.',
        f'Policy {POLICY["status"]}; selective evidence {selective}.',
        'Lihat stress dan benchmark pada evaluation_report.md. Kualitas internal dan kelayakan lapangan dinilai terpisah.']
    (root/'model_summary.md').write_text('\n\n'.join(summary),encoding='utf-8')
    training_lines=['# Training report',f'Jumlah run penuh selesai: {len(selection["all_runs"])} / 8.',
        selection['reason'],pd.read_csv(OUT/'training'/'experiments.csv').to_string(index=False),
        f'Finalis/agregat: {selection["aggregates"]}',
        'Pemilihan memakai validation; checkpoint handoff seed42. Test tidak dipakai training. '
        'Run opsional sampler/full-finetune/brightness/noise/CLAHE tidak dijalankan pada suite default.']
    (OUT/'training'/'training_report.md').write_text('\n\n'.join(training_lines),encoding='utf-8')
    files=[p for p in root.rglob('*') if p.is_file() and p.name!='artifact_checksums.json']
    save_json(root/'artifact_checksums.json',{str(p.relative_to(root)):sha256_file(p) for p in files})
    return assessment

ASSESSMENT=None
if RUN_MODE=='full':
    ASSESSMENT=write_reports(SELECTION,TEST_METRICS,TEST_CI,STRESS,HOST_BENCHMARK)
    display(Markdown((OUT/'evaluation'/'model_summary.md').read_text()))
    print('Execution:',ASSESSMENT['execution_status'],'| Evidence:',ASSESSMENT['evidence_scope'])
    print('Artefak tersimpan:',OUT)


Model terpilih: E03_large_seed42.

Recall terendah: pestalotiopsis. Kasus sakit→healthy: {'value': 0.00819672131147541, 'numerator': 1, 'denominator': 122, 'unit': 'proportion', 'status': 'defined'}.

Policy feasible; selective evidence {'coverage': {'value': 0.9925925925925926, 'numerator': 134, 'denominator': 135, 'unit': 'proportion', 'status': 'defined'}, 'selective_risk': {'value': 0.007462686567164179, 'numerator': 1, 'denominator': 134, 'unit': 'proportion', 'status': 'defined'}, 'correct_accepted_yield': {'value': 0.9851851851851852, 'numerator': 133, 'denominator': 135, 'unit': 'proportion', 'status': 'defined'}, 'false_reassurance': {'value': 0.0, 'numerator': 0, 'denominator': 122, 'unit': 'proportion', 'status': 'defined'}, 'conditional_healthy_risk': {'value': 0.0, 'numerator': 0, 'denominator': 13, 'unit': 'proportion', 'status': 'defined'}, 'accepted_predicted_healthy': 13, 'accepted_total': 134, 'per_class': {'0': {'coverage': {'value': 1.0, 'numerator': 25, 'denominator': 25, 'unit': 'proportion', 'status': 'defined'}, 'accepted_recall': {'value': 1.0, 'numerator': 25, 'denominator': 25, 'unit': 'proportion', 'status': 'defined'}, 'refusal_rate': {'value': 0.0, 'numerator': 0, 'denominator': 25, 'unit': 'proportion', 'status': 'defined'}, 'accepted_true_support': 25, 'accepted_predicted_support': 25}, '1': {'coverage': {'value': 1.0, 'numerator': 13, 'denominator': 13, 'unit': 'proportion', 'status': 'defined'}, 'accepted_recall': {'value': 1.0, 'numerator': 13, 'denominator': 13, 'unit': 'proportion', 'status': 'defined'}, 'refusal_rate': {'value': 0.0, 'numerator': 0, 'denominator': 13, 'unit': 'proportion', 'status': 'defined'}, 'accepted_true_support': 13, 'accepted_predicted_support': 13}, '2': {'coverage': {'value': 0.9615384615384616, 'numerator': 25, 'denominator': 26, 'unit': 'proportion', 'status': 'defined'}, 'accepted_recall': {'value': 0.9615384615384616, 'numerator': 25, 'denominator': 26, 'unit': 'proportion', 'status': 'defined'}, 'refusal_rate': {'value': 0.038461538461538464, 'numerator': 1, 'denominator': 26, 'unit': 'proportion', 'status': 'defined'}, 'accepted_true_support': 25, 'accepted_predicted_support': 26}, '3': {'coverage': {'value': 1.0, 'numerator': 71, 'denominator': 71, 'unit': 'proportion', 'status': 'defined'}, 'accepted_recall': {'value': 0.9859154929577465, 'numerator': 70, 'denominator': 71, 'unit': 'proportion', 'status': 'defined'}, 'refusal_rate': {'value': 0.0, 'numerator': 0, 'denominator': 71, 'unit': 'proportion', 'status': 'defined'}, 'accepted_true_support': 71, 'accepted_predicted_support': 70}}, 'decision_counts': {'correct_accepted': 133, 'wrong_disease_name': 1, 'disease_to_healthy': 0, 'healthy_to_disease': 0, 'uncertain': 1, 'technical_error': 0}}.

Lihat stress dan benchmark pada evaluation_report.md. Kualitas internal dan kelayakan lapangan dinilai terpisah.

Execution: completed | Evidence: internal_only
Artefak tersimpan: /kaggle/working/outputs/bananalsd_notebook_device_auto_v2


## Validasi implementasi saat penyusunan — 7 November 2026

Pemeriksaan berikut benar-benar dijalankan pada CPU dengan PyTorch `2.9.1+cpu` dan
TorchVision `0.24.1+cpu`:

- Schema notebook dan kompilasi seluruh sel Python.
- Loader/audit pada 937 baris raw sebenarnya, karantina konflik, deduplikasi, EDA,
  augmentation preview, dan split 627/135/135 tanpa overlap ID/hash/grup.
- Forward/backward MobileNetV3 Small dan Large, aturan freeze tahap A/B,
  serta verifikasi buffer BatchNorm tidak berubah.
- Sanity pretrained pada 16 gambar train, 200 update; model diagnostik terpisah.
- Smoke training+validation data asli dengan inisialisasi acak untuk menguji loop,
  weighted-loss denominator, checkpoint, dan plot (bukan eksperimen transfer learning final).
- Resume batas epoch menghasilkan bobot identik dengan run tanpa interupsi;
  checksum menolak checkpoint cache yang dimodifikasi.
- Kasus metrik denominator nol, abstention semua, support policy terlalu kecil,
  tidak ada policy layak, penolakan fitting pada test, temperature positif/argmax tetap,
  dan CI untuk nol error/interval bootstrap degenerat.
- Integrasi evaluator pada **fixture sintetis terpisah**: 25 fixture konsistensi,
  save/load dan batch-size equivalence, protocol/code fingerprint guards, 12 profil
  stress, gallery, benchmark host, dan pembuatan laporan.

**Belum dijalankan:** delapan run penuh, seleksi model penelitian final, inference pada
held-out test BananaLSD sebenarnya, atau validasi lapangan/OOD nyata. Tidak ada skor
fixture yang disajikan sebagai hasil penelitian. Sel kode notebook dibiarkan tanpa
output eksperimen agar Run All mencatat hasil dari environment dan konfigurasi Anda sendiri.


## 19. Peta artefak dan status eksekusi

```text
outputs/bananalsd_notebook_device_auto_v2/
  config.resolved.json, environment.json, dependency-lock.txt
  label_mapping.json, preprocessing.resolved.json
  cache/{hf,rgb}/                         # salinan kerja, sumber asli tetap
  eda/, augmentation_samples/, reports/
  manifests/{train,validation,test}.csv
  diagnostics/{sanity,SMOKE_small_seed42}/
  training/
    experiments.csv, selection.json, training_report.md
    <run_id>/{metrics.csv,metrics.json,summary.json,provenance.json,...}
    <run_id>/checkpoints/{best_head,best,last}.pt
    <run_id>/validation/{logits.csv,metrics.json,confusion_matrix*.png}
    handoff/{model_state_dict.pt,model_config.json,label_mapping.json,...}
  evaluation/
    protocol.lock.json, test_access_log.jsonl, validation_logits.csv
    consistency/, calibration/, policy/
    internal/{predictions.csv,metrics.json,confidence_intervals.json,...}
    stress/{manifest.csv,metrics.csv,paired_deltas.json,predictions/}
    errors/{index.csv,gallery/}
    efficiency/{raw_timings.csv,host_benchmark.json,memory.json}
    evaluation_report.md, model_card.md, model_summary.md
    model_assessment.json, artifact_checksums.json
```

Notebook memakai JSON resolved config sebagai pengganti YAML terpisah agar mandiri.
Tidak menulis ulang README rencana. Tidak membuat CLI/package yang belum diminta.
Untuk model baru sesudah test dilihat, pilih direktori output versi baru; test lama
tetap pernah dilihat, sehingga diperlukan holdout baru untuk klaim konfirmatori baru.

## Referensi implementasi

- [Dataset card dan label BananaLSD](https://huggingface.co/datasets/Project-AgML/BananaLSD_leaf_disease_classification).
- [MobileNetV3-Small](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.mobilenet_v3_small.html),
  [MobileNetV3-Large](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.mobilenet_v3_large.html).
- [PyTorch reproducibility](https://docs.pytorch.org/docs/stable/notes/randomness.html),
  [serialization](https://docs.pytorch.org/docs/stable/notes/serialization.html).
- [BananaLSD, Arman et al. 2023](https://doi.org/10.1016/j.dib.2023.109608).
- [Temperature scaling, Guo et al.](https://proceedings.mlr.press/v70/guo17a.html).
- [Selective classification](https://arxiv.org/abs/1705.08500).

Referensi rencana lokal: `Model/Pre-processing/README.md`, `Model/Training/README.md`,
`Model/Evaluate/README.md`. Numerical training/augmentation/threshold settings adalah
keputusan proyek dari rencana tersebut; bukan parameter MobileNetV3 dari publikasi dataset.
